# Melanoma: combined evaluation, confidence intervals, comparisons and app

**Google Colab → Runtime → Change runtime type → GPU → T4. Run cells in order.**

This notebook combines the complete Attention U-Net + EfficientNetB0 reporting workflow with the confidence-interval and earlier-run comparison supplement. No separate reporting notebook is needed.

Default `MODE = "evaluate_saved"` loads the frozen 7 October candidate from Google Drive. It retains the saved models, preprocessing, mask inference and validation-selected classifier threshold. The model files and configuration must still exist at `SAVED_RUN_DIR` in Section 3.

The workflow produces segmentation and classification test loss, Dice/IoU, classification metrics and figures, best/median/worst examples, ROI and Grad-CAM comparisons, training-history evidence, **95% confidence intervals**, optional paired earlier-run comparisons, a Results paragraph, an HTML report, a deployment ZIP and the Gradio app. All outputs use the **same newly generated run folder**. Confidence-interval tables and comparison outputs are included in the report and ZIP.

Set bootstrap options and optional `PRIOR_RUN_DIRS` in Section 3 before running. Earlier runs need compatible per-image CSVs and their own saved classifier threshold. Paired deltas require identical image IDs and diagnosis labels; matching IDs alone does not establish a controlled experiment. Check reference masks, training splits and preprocessing when interpreting model differences.

Historical curves recovered from earlier notebooks are labelled as rounded console summaries. They are not substitutes for exact per-epoch histories. `MODE = "train_new"` remains available only for a separate new experiment; exact histories are then saved automatically.

All official-test results remain exploratory follow-up because this test set has already been inspected. Intervals quantify sampling uncertainty for this dataset and do not remove test-set reuse, model-selection bias or within-patient dependence. The supplied analysis resamples images; patient-level intervals require patient IDs and a separate cluster-bootstrap analysis.

The combined notebook has been checked for structure, code syntax and supplemental calculations using synthetic fixtures. Its full model evaluation must be run in Colab with your saved weights and data; no new model results are prefilled here.


## 1. Install the software interface

In [ ]:
# Install only the UI dependency; keep Colab's existing TensorFlow runtime.
%pip install -q "gradio==6.30.0"


## 2. Imports and embedded software modules

In [ ]:
import os
import gc
import json
import random
import shutil
import time
import zipfile
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


In [ ]:
import sys
from pathlib import Path
MODULE_DIR = Path("/content/attention_melanoma_modules")
MODULE_DIR.mkdir(parents=True, exist_ok=True)
(MODULE_DIR / "attention_pipeline_core.py").write_text('"""Inference and Gradio UI for the frozen Attention U-Net melanoma candidate.\n\nNo training or threshold selection occurs in this module. JPEG ROI encoding is\npreserved because the training notebook classified saved quality-95 JPEG ROIs.\n"""\nfrom __future__ import annotations\n\nfrom io import BytesIO\nfrom pathlib import Path\nimport hashlib\nimport json\nimport threading\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nfrom scipy import ndimage\nimport tensorflow as tf\n\nSEG_SIZE = (256, 256)\nCLS_SIZE = (224, 224)\n\n\ndef image_tensor(path, size, normalise=False):\n    pixels = tf.io.decode_image(tf.io.read_file(str(path)), channels=3,\n                                expand_animations=False)\n    pixels = tf.cast(tf.image.resize(pixels, size, method="bilinear"), tf.float32)\n    return pixels / 255.0 if normalise else pixels\n\n\ndef roi_tensor(roi):\n    # Match roi.save(..., quality=95) -> decode_classifier_image() in training.\n    buffer = BytesIO()\n    roi.convert("RGB").save(buffer, format="JPEG", quality=95)\n    pixels = tf.io.decode_jpeg(buffer.getvalue(), channels=3)\n    return tf.cast(tf.image.resize(pixels, CLS_SIZE), tf.float32)\n\n\ndef crop_from_prediction(image_pil, predicted_mask, padding=0.15):\n    image = image_pil.convert("RGB")\n    width, height = image.size\n    binary = Image.fromarray(np.uint8(np.asarray(predicted_mask) > 0) * 255)\n    binary = np.asarray(binary.resize(image.size, Image.Resampling.NEAREST)) > 0\n    labelled, count = ndimage.label(binary)\n    reason = "cropped"\n    if count == 0:\n        crop, reason = image, "empty predicted mask"\n    else:\n        sizes = np.bincount(labelled.ravel())\n        sizes[0] = 0\n        lesion = labelled == int(sizes.argmax())\n        area = int(lesion.sum())\n        if area < max(64, int(width * height * 0.001)) or area > width * height * 0.80:\n            crop, reason = image, "implausible predicted area"\n        else:\n            ys, xs = np.where(lesion)\n            span = max(np.ptp(xs) + 1, np.ptp(ys) + 1)\n            margin = int(np.ceil(span * padding))\n            x0 = max(0, int(xs.min()) - margin)\n            x1 = min(width, int(xs.max()) + margin + 1)\n            y0 = max(0, int(ys.min()) - margin)\n            y1 = min(height, int(ys.max()) + margin + 1)\n            crop = image.crop((x0, y0, x1, y1))\n    a = np.asarray(crop)\n    h, w = a.shape[:2]\n    side = max(h, w)\n    top, left = (side - h) // 2, (side - w) // 2\n    square = np.pad(a, ((top, side - h - top), (left, side - w - left), (0, 0)),\n                    mode="edge")\n    roi = Image.fromarray(square).resize((CLS_SIZE[1], CLS_SIZE[0]),\n                                         Image.Resampling.BILINEAR)\n    return roi, reason\n\n\ndef mask_overlay(image, mask):\n    base = np.asarray(image.convert("RGB"), dtype=np.float32)\n    mask = Image.fromarray(np.uint8(mask) * 255).resize(\n        image.size, Image.Resampling.NEAREST)\n    binary = np.asarray(mask) > 127\n    tint = np.array([35, 195, 145], dtype=np.float32)\n    base[binary] = 0.65 * base[binary] + 0.35 * tint\n    boundary = binary & ~ndimage.binary_erosion(binary)\n    base[boundary] = [255, 215, 30]\n    return Image.fromarray(np.uint8(np.clip(base, 0, 255)))\n\n\ndef gradcam_overlay(roi, heatmap):\n    from matplotlib import colormaps\n    image = roi.convert("RGB").resize((CLS_SIZE[1], CLS_SIZE[0]))\n    heat = Image.fromarray(np.uint8(np.clip(heatmap, 0, 1) * 255)).resize(\n        image.size, Image.Resampling.BILINEAR)\n    colors = colormaps["jet"](np.asarray(heat, dtype=np.float32) / 255.0)[..., :3]\n    base = np.asarray(image, dtype=np.float32) / 255.0\n    return Image.fromarray(np.uint8(255 * np.clip(0.62 * base + 0.38 * colors, 0, 1)))\n\n\nclass MelanomaPipeline:\n    def __init__(self, segmenter, classifier, configuration, run_dir=None):\n        self.segmenter = segmenter\n        self.classifier = classifier\n        self.configuration = dict(configuration)\n        self.run_dir = Path(run_dir) if run_dir else None\n        self.lock = threading.RLock()\n        cfg = self.configuration\n        if cfg.get("view") != "dual":\n            raise ValueError("This app requires the full-image + ROI classifier (view=\'dual\').")\n        for field in ("threshold", "segmentation_threshold"):\n            if field not in cfg or not 0 <= float(cfg[field]) <= 1:\n                raise ValueError(f"Missing or invalid saved configuration field: {field}")\n        for field in ("method", "segmentation_method"):\n            if cfg.get(field) not in ("single", "4-flip average"):\n                raise ValueError(f"Unsupported inference method: {cfg.get(field)}")\n        if segmenter.input_shape[1:] != (*SEG_SIZE, 3):\n            raise ValueError("Segmenter input must be 256 x 256 RGB.")\n        if len(classifier.inputs) != 2:\n            raise ValueError("Expected two classifier inputs: full image and ROI.")\n        shapes = [tuple(x.shape[1:]) for x in classifier.inputs]\n        if shapes != [(*CLS_SIZE, 3), (*CLS_SIZE, 3)]:\n            raise ValueError(f"Classifier input shapes are incompatible: {shapes}")\n\n    @classmethod\n    def load(cls, run_dir):\n        run_dir = Path(run_dir)\n        paths = {\n            "configuration": run_dir / "tables" / "selected_configuration.json",\n            "segmenter": run_dir / "models" / "attention_unet_best.keras",\n            "classifier": run_dir / "models" / "selected_final_classifier.keras",\n        }\n        missing = [str(p) for p in paths.values() if not p.is_file()]\n        if missing:\n            raise FileNotFoundError("The saved pipeline is incomplete. Missing:\\n" + "\\n".join(missing))\n        configuration = json.loads(paths["configuration"].read_text())\n        provenance = run_dir / "tables" / "segmentation_configuration.json"\n        if provenance.is_file():\n            expected = json.loads(provenance.read_text()).get("checkpoint_sha256")\n            if expected:\n                actual = hashlib.sha256(paths["segmenter"].read_bytes()).hexdigest()\n                if actual != expected:\n                    raise ValueError("The segmentation checkpoint does not match its saved configuration.")\n        manifest_path = run_dir / "deployment_manifest.json"\n        if manifest_path.is_file():\n            for relative, expected in json.loads(manifest_path.read_text()).get("sha256", {}).items():\n                actual = hashlib.sha256((run_dir / relative).read_bytes()).hexdigest()\n                if actual != expected:\n                    raise ValueError(f"Deployment bundle file has changed: {relative}")\n        # compile=False avoids needing custom training losses for inference.\n        segmenter = tf.keras.models.load_model(paths["segmenter"], compile=False)\n        classifier = tf.keras.models.load_model(paths["classifier"], compile=False)\n        return cls(segmenter, classifier, configuration, run_dir)\n\n    def segmentation_probabilities(self, images, batch_size=4):\n        images = np.asarray(images, dtype=np.float32)\n        methods = [(), (2,), (1,), (1, 2)] if self.configuration[\n            "segmentation_method"] == "4-flip average" else [()]\n        predictions = []\n        for axes in methods:\n            transformed = np.flip(images, axis=axes) if axes else images\n            score = self.segmenter.predict(np.ascontiguousarray(transformed),\n                                            batch_size=batch_size, verbose=0)[..., 0]\n            score = np.flip(score, axis=axes) if axes else score\n            predictions.append(score)\n        return np.mean(np.stack(predictions), axis=0)\n\n    def classifier_probabilities(self, full_images, roi_images, batch_size=8):\n        full_images = np.asarray(full_images, dtype=np.float32)\n        roi_images = np.asarray(roi_images, dtype=np.float32)\n        methods = [(), (2,), (1,), (1, 2)] if self.configuration[\n            "method"] == "4-flip average" else [()]\n        predictions = []\n        for axes in methods:\n            full = np.flip(full_images, axis=axes) if axes else full_images\n            roi = np.flip(roi_images, axis=axes) if axes else roi_images\n            features = {"full_image": np.ascontiguousarray(full),\n                        "roi_image": np.ascontiguousarray(roi)}\n            predictions.append(self.classifier.predict(\n                features, batch_size=batch_size, verbose=0).ravel())\n        return np.mean(np.stack(predictions), axis=0)\n\n    def gradcam(self, full_batch, roi_batch, target_class):\n        model = self.classifier\n        backbone = model.get_layer("efficientnetb0")\n        gap = model.get_layer("gap")\n        with tf.GradientTape() as tape:\n            full_map = backbone(full_batch, training=False)\n            roi_map = backbone(roi_batch, training=False)\n            tape.watch(roi_map)\n            features = model.get_layer("combine_features")([gap(full_map), gap(roi_map)])\n            hidden = model.get_layer("hidden_dense")(features)\n            hidden = model.get_layer("head_dropout")(hidden, training=False)\n            output = model.get_layer("prediction")\n            logit = tf.linalg.matmul(hidden, output.kernel) + output.bias\n            score = logit[:, 0] if target_class == 1 else -logit[:, 0]\n        gradients = tape.gradient(score, roi_map)\n        if gradients is None:\n            raise RuntimeError("Grad-CAM could not trace the ROI feature map.")\n        weights = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)\n        heatmap = tf.nn.relu(tf.reduce_sum(roi_map * weights, axis=-1)[0])\n        return (heatmap / (tf.reduce_max(heatmap) + 1e-8)).numpy()\n\n    def predict_file(self, path, explain=True):\n        if path is None:\n            raise ValueError("Upload a dermoscopic lesion image first.")\n        path = Path(path)\n        with Image.open(path) as handle:\n            original = handle.convert("RGB")\n        with self.lock:\n            segmentation_input = image_tensor(path, SEG_SIZE, normalise=True)[None, ...]\n            probabilities = self.segmentation_probabilities(segmentation_input)[0]\n            mask = probabilities >= float(self.configuration["segmentation_threshold"])\n            roi, reason = crop_from_prediction(\n                original, mask, padding=float(self.configuration.get("roi_padding", 0.15)))\n            full_input = image_tensor(path, CLS_SIZE)[None, ...]\n            roi_input = roi_tensor(roi)[None, ...]\n            score = float(self.classifier_probabilities(full_input, roi_input)[0])\n            predicted = int(score >= float(self.configuration["threshold"]))\n            heatmap = self.gradcam(full_input, roi_input, predicted) if explain else None\n        result = {\n            "original": original, "mask": Image.fromarray(np.uint8(mask) * 255),\n            "mask_overlay": mask_overlay(original, mask), "roi": roi,\n            "score": score, "predicted_label": predicted,\n            "label": "Melanoma" if predicted else "Benign",\n            "roi_reason": reason, "probability_map": probabilities,\n        }\n        result["gradcam"] = gradcam_overlay(roi, heatmap) if heatmap is not None else None\n        return result\n\n\ndef build_demo(pipeline, sample_paths=()):\n    import gradio as gr\n    cfg = pipeline.configuration\n\n    def preview(path):\n        if path is None:\n            return None\n        with Image.open(path) as image:\n            return image.convert("RGB")\n\n    def analyse(path):\n        try:\n            result = pipeline.predict_file(path)\n        except (ValueError, FileNotFoundError, OSError) as error:\n            raise gr.Error(str(error)) from error\n        summary = f"### Model prediction: **{result[\'label\']}**\\n"\n        summary += (f"Melanoma model score: **{result[\'score\']:.4f}**. "\n                    f"Decision threshold: {float(cfg[\'threshold\']):.4f}.\\n\\n"\n                    "The score is a model output, not a calibrated clinical probability.")\n        if result["roi_reason"] != "cropped":\n            summary += "\\n\\nThe lesion crop was unreliable; the classifier used the full image for the ROI view."\n        panels = [(result["original"], "Original image"),\n                  (result["mask"], "Predicted lesion mask"),\n                  (result["mask_overlay"], "Lesion outline"),\n                  (result["roi"], "Lesion ROI"),\n                  (result["gradcam"], "Grad-CAM on ROI")]\n        return summary, panels\n\n    def clear():\n        return None, None, "Upload a dermoscopic lesion image, then select Analyse.", []\n\n    with gr.Blocks(title="Attention U-Net melanoma research demo",\n                   delete_cache=(3600, 3600)) as demo:\n        gr.Markdown("# Melanoma image analysis\\n"\n                    "Attention U-Net lesion segmentation with EfficientNetB0 classification.\\n\\n"\n                    "**Student research prototype. Not validated for clinical diagnosis.**")\n        with gr.Tab("Analyse image"):\n            with gr.Row():\n                with gr.Column(scale=1):\n                    upload = gr.File(label="Upload dermoscopic image (JPG or PNG)",\n                                     file_types=[".jpg", ".jpeg", ".png"], type="filepath")\n                    input_preview = gr.Image(label="Uploaded image", interactive=False)\n                    with gr.Row():\n                        analyse_button = gr.Button("Analyse", variant="primary")\n                        clear_button = gr.Button("Clear")\n                    if sample_paths:\n                        gr.Examples(examples=[[str(p)] for p in sample_paths],\n                                    inputs=upload, label="Public ISIC example images")\n                with gr.Column(scale=2):\n                    summary = gr.Markdown("Upload a dermoscopic lesion image, then select Analyse.")\n                    panels = gr.Gallery(label="Pipeline outputs", columns=3,\n                                        object_fit="contain", height="auto")\n                    gr.Markdown("The lesion mask estimates the lesion boundary. "\n                                "Grad-CAM shows image regions contributing to the model prediction; "\n                                "it is not a lesion mask.")\n            upload.change(preview, inputs=upload, outputs=input_preview)\n            analyse_button.click(analyse, inputs=upload, outputs=[summary, panels],\n                                 api_name="analyse", concurrency_limit=1)\n            clear_button.click(clear, outputs=[upload, input_preview, summary, panels])\n        with gr.Tab("Evaluation"):\n            gr.Markdown("## Saved evaluation results\\n"\n                        "Classification and segmentation measure different tasks. "\n                        "The official ISIC test set was viewed in earlier experiments; "\n                        "these results are exploratory follow-up.")\n            run_dir = pipeline.run_dir\n            displayed = False\n            if run_dir:\n                for filename, label in [("complete_segmentation_test_metrics.csv", "Lesion segmentation"),\n                                        ("complete_classification_test_metrics.csv", "Melanoma classification")]:\n                    path = run_dir / "tables" / filename\n                    if path.is_file():\n                        gr.Markdown("### " + label)\n                        gr.Dataframe(value=pd.read_csv(path).T.reset_index(),\n                                     headers=["Metric", "Value"], interactive=False)\n                        displayed = True\n                figures = sorted((run_dir / "figures").glob("*.png"))\n                if figures:\n                    gr.Gallery(value=[(str(p), p.stem.replace("_", " ")) for p in figures],\n                               label="Evaluation figures", columns=1, object_fit="contain")\n                    displayed = True\n            if not displayed:\n                gr.Markdown("Run the reporting notebook to add test loss, metrics and evaluation figures.")\n    return demo\n', encoding="utf-8")
(MODULE_DIR / "attention_reporting.py").write_text('"""Complete evaluation and figures for a locked Attention U-Net pipeline."""\nfrom pathlib import Path\nimport base64\nimport html\nimport json\nimport shutil\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom PIL import Image\nimport tensorflow as tf\nfrom sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve\n\nfrom attention_pipeline_core import (\n    MelanomaPipeline, image_tensor, mask_overlay, SEG_SIZE,\n)\n\nEPS = 1e-7\n\n\ndef truth_mask(path):\n    pixels = tf.io.decode_png(tf.io.read_file(str(path)), channels=1)\n    pixels = tf.image.resize(pixels, SEG_SIZE, method="nearest")\n    return (pixels.numpy()[..., 0] > 127)\n\n\ndef segmentation_rows(pipeline, frame, batch_size=12):\n    rows = []\n    start_time = time.perf_counter()\n    threshold = float(pipeline.configuration["segmentation_threshold"])\n    for begin in range(0, len(frame), batch_size):\n        batch = frame.iloc[begin:begin + batch_size]\n        images = np.stack([image_tensor(p, SEG_SIZE, normalise=True).numpy()\n                           for p in batch.image_path])\n        truth = np.stack([truth_mask(p) for p in batch.mask_path])\n        probability = pipeline.segmentation_probabilities(images)\n        y_true = truth.astype(np.float32)\n        # Loss is evaluated on probabilities BEFORE thresholding.\n        bce = tf.keras.losses.binary_crossentropy(\n            tf.convert_to_tensor(y_true[..., None]),\n            tf.convert_to_tensor(probability[..., None])).numpy().mean(axis=(1, 2))\n        soft_dice = (2 * (y_true * probability).sum(axis=(1, 2)) + EPS) / (\n            y_true.sum(axis=(1, 2)) + probability.sum(axis=(1, 2)) + EPS)\n        prediction = probability >= threshold\n        tp = (truth & prediction).sum(axis=(1, 2))\n        tn = (~truth & ~prediction).sum(axis=(1, 2))\n        fp = (~truth & prediction).sum(axis=(1, 2))\n        fn = (truth & ~prediction).sum(axis=(1, 2))\n        dice = (2 * tp + EPS) / (2 * tp + fp + fn + EPS)\n        iou = (tp + EPS) / (tp + fp + fn + EPS)\n        for j, record in enumerate(batch.itertuples(index=False)):\n            rows.append({\n                "image_id": record.image_id, "dice": float(dice[j]),\n                "iou": float(iou[j]), "bce": float(bce[j]),\n                "soft_dice": float(soft_dice[j]),\n                "test_loss": float(bce[j] + 1.0 - soft_dice[j]),\n                "pixel_TP": int(tp[j]), "pixel_TN": int(tn[j]),\n                "pixel_FP": int(fp[j]), "pixel_FN": int(fn[j]),\n            })\n        if begin == 0 or begin + len(batch) == len(frame) or (begin + len(batch)) % 120 == 0:\n            print(f"Segmentation evaluation: {begin + len(batch)}/{len(frame)}")\n    return pd.DataFrame(rows), time.perf_counter() - start_time\n\n\ndef classification_metrics(labels, probabilities, threshold):\n    truth = np.asarray(labels, dtype=int)\n    probabilities = np.asarray(probabilities, dtype=np.float64).reshape(-1)\n    predicted = (probabilities >= threshold).astype(int)\n    tn, fp, fn, tp = confusion_matrix(truth, predicted, labels=[0, 1]).ravel()\n    sensitivity = float(tp / (tp + fn)) if tp + fn else np.nan\n    specificity = float(tn / (tn + fp)) if tn + fp else np.nan\n    precision = float(tp / (tp + fp)) if tp + fp else 0.0\n    f1 = float(2 * tp / (2 * tp + fp + fn)) if 2 * tp + fp + fn else 0.0\n    clipped = np.clip(probabilities, EPS, 1 - EPS)\n    bce = float(np.mean(-truth * np.log(clipped) - (1 - truth) * np.log(1 - clipped)))\n    return {\n        "N": len(truth), "Melanoma cases": int(truth.sum()),\n        "Benign cases": int((truth == 0).sum()),\n        "Accuracy": float((tp + tn) / len(truth)),\n        "Sensitivity": sensitivity, "Specificity": specificity,\n        "Precision": precision, "F1": f1,\n        "Balanced accuracy": float((sensitivity + specificity) / 2),\n        "ROC-AUC": float(roc_auc_score(truth, probabilities)),\n        "Classification test loss (unweighted BCE)": bce,\n        "Threshold": float(threshold),\n        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),\n    }\n\n\ndef save_figure(fig, output_dir, name):\n    output = Path(output_dir) / "figures" / f"{name}.png"\n    output.parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(output, dpi=200, bbox_inches="tight", facecolor="white")\n    plt.show()\n    plt.close(fig)\n    return output\n\n\ndef plot_training_history(frame, output_dir, name, title, segmentation=False,\n                          provenance="Exact saved training history"):\n    if frame is None or len(frame) == 0:\n        return None\n    frame = frame.copy()\n    x = frame["epoch"].to_numpy() if "epoch" in frame else np.arange(1, len(frame) + 1)\n    if len(x) and x.min() == 0:\n        x = x + 1  # CSVLogger records epochs from zero.\n    pairs = [("loss", "val_loss", "Loss (BCE + soft Dice loss)"),\n             ("dice_coefficient", "val_dice_coefficient", "Hard Dice at threshold 0.5"),\n             ("iou_coefficient", "val_iou_coefficient", "IoU / Jaccard at threshold 0.5")]\n    if not segmentation:\n        pairs = [("loss", "val_loss", "Loss (binary cross-entropy)"),\n                 ("accuracy", "val_accuracy", "Binary accuracy at threshold 0.5"),\n                 ("auc", "val_auc", "Keras ROC-AUC")]\n    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))\n    for ax, (train_key, val_key, metric_title) in zip(axes, pairs):\n        if train_key in frame:\n            ax.plot(x, frame[train_key], label="Training", color="#1c7197")\n        if val_key in frame:\n            ax.plot(x, frame[val_key], "--", label="Validation", color="#e3953f")\n        ax.set_xlabel("Epoch"); ax.set_title(metric_title, fontsize=11)\n        ax.grid(alpha=0.18); ax.legend(fontsize=9)\n        if train_key != "loss":\n            ax.set_ylim(0, 1)\n    fig.suptitle(title, fontsize=15)\n    note = provenance\n    if not segmentation:\n        note += ". Training loss uses class weights; validation loss does not."\n    fig.text(0.5, 0.01, note, ha="center", fontsize=8)\n    fig.tight_layout(rect=(0, 0.05, 1, 0.94))\n    return save_figure(fig, output_dir, name)\n\n\ndef segmentation_comparison(pipeline, frame, per_image, output_dir):\n    ordered = per_image.sort_values(["dice", "image_id"]).reset_index(drop=True)\n    samples = [("Best", ordered.iloc[-1]),\n               ("Median", ordered.iloc[len(ordered) // 2]),\n               ("Worst", ordered.iloc[0])]\n    fig, axes = plt.subplots(3, 5, figsize=(16, 10))\n    columns = ["Original image", "Expert lesion mask", "Predicted lesion mask", "Mask overlay", "Lesion ROI"]\n    for i, (position, metrics) in enumerate(samples):\n        record = frame.loc[frame.image_id == metrics.image_id].iloc[0]\n        result = pipeline.predict_file(record.image_path, explain=False)\n        truth = truth_mask(record.mask_path)\n        panels = [result["original"], Image.fromarray(np.uint8(truth) * 255),\n                  result["mask"], result["mask_overlay"], result["roi"]]\n        for j, panel in enumerate(panels):\n            axes[i, j].imshow(panel, cmap="gray" if j in (1, 2) else None)\n            axes[i, j].axis("off")\n            if i == 0:\n                axes[i, j].set_title(columns[j], fontsize=10)\n        axes[i, 0].text(0.5, -0.05,\n                        f"{position}: {metrics.image_id}\\nDice {metrics.dice:.4f} | IoU {metrics.iou:.4f}",\n                        transform=axes[i, 0].transAxes, ha="center", va="top", fontsize=9)\n    fig.suptitle("Attention U-Net: best, median and worst official-test masks", fontsize=15)\n    fig.text(0.5, 0.01, "Cases ranked by per-image Dice. Mask metrics computed at 256 x 256. "\n             "An empty predicted mask remains a segmentation failure even when ROI fallback is used.",\n             ha="center", fontsize=8)\n    fig.tight_layout(rect=(0, 0.045, 1, 0.95), h_pad=3)\n    return save_figure(fig, output_dir, "best_median_worst_masks")\n\n\ndef classification_figure(labels, probabilities, threshold, output_dir):\n    predicted = np.asarray(probabilities) >= threshold\n    matrix = confusion_matrix(labels, predicted, labels=[0, 1])\n    fpr, tpr, _ = roc_curve(labels, probabilities)\n    auc = roc_auc_score(labels, probabilities)\n    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))\n    axes[0].imshow(matrix, cmap="Blues")\n    axes[0].set_xticks([0, 1], labels=["Benign", "Melanoma"])\n    axes[0].set_yticks([0, 1], labels=["Benign", "Melanoma"])\n    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Expert label")\n    axes[0].set_title(f"Confusion matrix | threshold {threshold:.4f}")\n    for i in range(2):\n        for j in range(2):\n            axes[0].text(j, i, str(matrix[i, j]), ha="center", va="center", fontsize=16,\n                         color="white" if matrix[i, j] > matrix.max() / 2 else "black")\n    axes[1].plot(fpr, tpr, color="#1c7197", label=f"ROC-AUC = {auc:.4f}")\n    axes[1].plot([0, 1], [0, 1], "--", color="gray")\n    axes[1].set_xlabel("False positive rate"); axes[1].set_ylabel("Sensitivity")\n    axes[1].set_title("Official-test ROC curve"); axes[1].legend(); axes[1].grid(alpha=0.18)\n    fig.suptitle("Melanoma classification: exploratory follow-up", fontsize=14)\n    fig.tight_layout()\n    return save_figure(fig, output_dir, "classification_confusion_and_roc")\n\n\ndef outcome_comparison(pipeline, frame, probabilities, output_dir):\n    truth = frame.label.to_numpy()\n    predicted = np.asarray(probabilities) >= float(pipeline.configuration["threshold"])\n    outcomes = np.select([(truth == 1) & predicted, (truth == 0) & ~predicted,\n                          (truth == 0) & predicted], ["TP", "TN", "FP"], default="FN")\n    chosen = [(name, np.flatnonzero(outcomes == name)[0])\n              for name in ("TP", "TN", "FP", "FN") if (outcomes == name).any()]\n    if not chosen:\n        return None\n    fig, axes = plt.subplots(len(chosen), 4, figsize=(13, 3.2 * len(chosen)), squeeze=False)\n    for i, (name, index) in enumerate(chosen):\n        record = frame.iloc[index]\n        result = pipeline.predict_file(record.image_path, explain=True)\n        panels = [result["original"], result["mask_overlay"], result["roi"], result["gradcam"]]\n        for j, (panel, title) in enumerate(zip(panels,\n                  ["Original", "Predicted lesion outline", "Lesion ROI", "Grad-CAM on ROI"])):\n            axes[i, j].imshow(panel); axes[i, j].axis("off")\n            axes[i, j].set_title(title, fontsize=10)\n        axes[i, 0].text(0.5, -0.06,\n                        f"{name}: {record.image_id} | melanoma score {result[\'score\']:.4f}",\n                        transform=axes[i, 0].transAxes, ha="center", va="top", fontsize=9)\n    fig.suptitle("Classification examples: TP, TN, FP and FN", fontsize=15)\n    fig.text(0.5, 0.005, "First image in dataset order for each available outcome. "\n             "Grad-CAM explains the predicted class for a single ROI view; it is not a lesion mask.",\n             ha="center", fontsize=8)\n    fig.tight_layout(rect=(0, 0.03, 1, 0.96), h_pad=3)\n    return save_figure(fig, output_dir, "classification_outcomes_and_gradcam")\n\n\ndef check_app_parity(pipeline, frame, notebook_probabilities):\n    # Include a near-threshold case, where deployment mistakes change decisions.\n    probabilities = np.asarray(notebook_probabilities)\n    threshold = float(pipeline.configuration["threshold"])\n    indices = list(dict.fromkeys([0, len(frame) // 2,\n                                  int(np.argmin(np.abs(probabilities - threshold)))]))\n    rows = []\n    for i in indices:\n        record = frame.iloc[i]\n        result = pipeline.predict_file(record.image_path, explain=False)\n        reference_mask = np.asarray(Image.open(record.pred_mask_path).convert("L")) > 127\n        mask_matches = bool(np.array_equal(np.asarray(result["mask"]) > 127, reference_mask))\n        score_matches = bool(np.isclose(result["score"], probabilities[i], rtol=1e-5, atol=1e-6))\n        decision_matches = bool(result["predicted_label"] == int(probabilities[i] >= threshold))\n        rows.append({"image_id": record.image_id, "Notebook score": float(probabilities[i]),\n                     "App score": result["score"], "Mask matches": mask_matches,\n                     "Score matches": score_matches, "Decision matches": decision_matches})\n    results = pd.DataFrame(rows)\n    if not results[["Mask matches", "Score matches", "Decision matches"]].to_numpy().all():\n        raise AssertionError("App outputs differ from the evaluated pipeline:\\n" + results.to_string(index=False))\n    return results\n\n\ndef evaluate_pipeline(pipeline, test_frame, notebook_probabilities, output_dir,\n                      validation_segmentation=None, make_figures=True, verify_parity=True):\n    output_dir = Path(output_dir)\n    tables_dir = output_dir / "tables"\n    tables_dir.mkdir(parents=True, exist_ok=True)\n    per_image, seconds = segmentation_rows(pipeline, test_frame)\n    per_image.to_csv(tables_dir / "complete_segmentation_per_image.csv", index=False)\n    tp, tn, fp, fn = [int(per_image[c].sum()) for c in ("pixel_TP", "pixel_TN", "pixel_FP", "pixel_FN")]\n    seg = {\n        "N": len(per_image), "Test loss (BCE + soft Dice loss)": float(per_image.test_loss.mean()),\n        "Mean BCE": float(per_image.bce.mean()),\n        "Mean soft Dice": float(per_image.soft_dice.mean()),\n        "Mean hard Dice": float(per_image.dice.mean()),\n        "Mean IoU / Jaccard": float(per_image.iou.mean()),\n        "Median Dice": float(per_image.dice.median()),\n        "Best Dice": float(per_image.dice.max()), "Worst Dice": float(per_image.dice.min()),\n        "Pixel sensitivity (lesion)": tp / (tp + fn) if tp + fn else np.nan,\n        "Pixel specificity (background)": tn / (tn + fp) if tn + fp else np.nan,\n        "Pixel TP": tp, "Pixel TN": tn, "Pixel FP": fp, "Pixel FN": fn,\n        "Mask threshold": float(pipeline.configuration["segmentation_threshold"]),\n        "Inference method": pipeline.configuration["segmentation_method"],\n        "Evaluation seconds": seconds, "Segmenter parameters": pipeline.segmenter.count_params(),\n        "Metric resolution": "256 x 256",\n    }\n    if validation_segmentation is not None and len(validation_segmentation):\n        selected = validation_segmentation.loc[\n            (validation_segmentation["Method"] == pipeline.configuration["segmentation_method"]) &\n            np.isclose(validation_segmentation["Threshold"].astype(float),\n                       float(pipeline.configuration["segmentation_threshold"]))]\n        if len(selected):\n            seg["Selected validation Dice"] = float(selected.iloc[0]["Validation Dice"])\n            seg["Selected validation IoU"] = float(selected.iloc[0]["Validation IoU"])\n    cls = classification_metrics(test_frame.label.to_numpy(), notebook_probabilities,\n                                 float(pipeline.configuration["threshold"]))\n    cls.update({"Classifier stage": pipeline.configuration["stage"],\n                "Inference method": pipeline.configuration["method"],\n                "Classifier parameters": pipeline.classifier.count_params()})\n    pd.DataFrame([seg]).to_csv(tables_dir / "complete_segmentation_test_metrics.csv", index=False)\n    pd.DataFrame([cls]).to_csv(tables_dir / "complete_classification_test_metrics.csv", index=False)\n    if verify_parity:\n        parity = check_app_parity(pipeline, test_frame, notebook_probabilities)\n        parity.to_csv(tables_dir / "app_notebook_parity.csv", index=False)\n        print("App/notebook parity passed for original, middle and near-threshold cases.")\n    if make_figures:\n        segmentation_comparison(pipeline, test_frame, per_image, output_dir)\n        classification_figure(test_frame.label.to_numpy(), notebook_probabilities,\n                              float(pipeline.configuration["threshold"]), output_dir)\n        outcome_comparison(pipeline, test_frame, notebook_probabilities, output_dir)\n        fig, ax = plt.subplots(figsize=(7, 4))\n        ax.hist(per_image.dice, bins=np.linspace(0, 1, 21), color="#1c7197", edgecolor="white")\n        ax.set_xlabel("Per-image Dice"); ax.set_ylabel("Number of test images")\n        ax.set_title("Attention U-Net: distribution of official-test Dice")\n        ax.grid(axis="y", alpha=0.18)\n        save_figure(fig, output_dir, "test_dice_distribution")\n    print("\\nSegmentation (BCE + soft Dice) test loss:", seg["Test loss (BCE + soft Dice loss)"])\n    print("Mean hard Dice:", seg["Mean hard Dice"], "| Mean IoU:", seg["Mean IoU / Jaccard"])\n    return seg, cls, per_image\n\n\ndef make_html_report(output_dir, segmentation_metrics, classifier_metrics, history_notes):\n    output_dir = Path(output_dir)\n    styles = "body{font:16px system-ui;max-width:1150px;margin:40px auto;padding:0 24px;color:#172f3d}" \\\n             "h1,h2{color:#155d75}table{border-collapse:collapse;width:100%;margin:20px 0}" \\\n             "td,th{border:1px solid #d2dfe4;text-align:left;padding:8px}img{width:100%}" \\\n             ".note{background:#eef5f7;padding:16px;border-radius:8px}"\n    chunks = ["<!doctype html><html><head><meta charset=\'utf-8\'>",\n              "<meta name=\'viewport\' content=\'width=device-width,initial-scale=1\'>",\n              f"<title>Attention U-Net evaluation</title><style>{styles}</style></head><body>",\n              "<h1>Attention U-Net + EfficientNetB0</h1>",\n              "<p class=\'note\'>Student research candidate. The official ISIC 2016 test set was previously viewed; "\n              "this is exploratory follow-up evaluation. Results do not establish clinical diagnostic validity.</p>",\n              "<h2>Lesion segmentation</h2>",\n              pd.DataFrame(list(segmentation_metrics.items()), columns=["Metric", "Value"]).to_html(index=False),\n              "<h2>Melanoma classification</h2>",\n              pd.DataFrame(list(classifier_metrics.items()), columns=["Metric", "Value"]).to_html(index=False),\n              "<h2>Training history provenance</h2><p>" + html.escape(history_notes) + "</p>",\n              "<p>Hard Dice uses thresholded masks. Test loss uses probability maps. "\n              "Pixel sensitivity/specificity measure lesion/background segmentation; classification "\n              "sensitivity/specificity measure melanoma/benign classification.</p>"]\n    for path in sorted((output_dir / "figures").glob("*.png")):\n        chunks.append("<h2>" + html.escape(path.stem.replace("_", " ")) + "</h2>")\n        encoded = base64.b64encode(path.read_bytes()).decode("ascii")\n        chunks.append(f"<img src=\'data:image/png;base64,{encoded}\' alt=\'{html.escape(path.stem)}\'>")\n    chunks.append("</body></html>")\n    result = output_dir / "evaluation_report.html"\n    result.write_text("\\n".join(chunks), encoding="utf-8")\n    return result\n', encoding="utf-8")
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
from attention_pipeline_core import MelanomaPipeline, build_demo
from attention_reporting import evaluate_pipeline, plot_training_history, make_html_report
RECOVERED_HISTORY = {'segmentation': [{'dice_coefficient': 0.7203, 'iou_coefficient': 0.6098, 'loss': 0.747, 'val_dice_coefficient': 0.6819, 'val_iou_coefficient': 0.564, 'val_loss': 1.0867, 'learning_rate': 0.001, 'epoch': 1}, {'dice_coefficient': 0.7814, 'iou_coefficient': 0.6753, 'loss': 0.5901, 'val_dice_coefficient': 0.6352, 'val_iou_coefficient': 0.5042, 'val_loss': 1.8769, 'learning_rate': 0.001, 'epoch': 2}, {'dice_coefficient': 0.8054, 'iou_coefficient': 0.7045, 'loss': 0.5214, 'val_dice_coefficient': 0.7068, 'val_iou_coefficient': 0.5889, 'val_loss': 0.9383, 'learning_rate': 0.001, 'epoch': 3}, {'dice_coefficient': 0.81, 'iou_coefficient': 0.7093, 'loss': 0.5076, 'val_dice_coefficient': 0.7595, 'val_iou_coefficient': 0.6482, 'val_loss': 1.0627, 'learning_rate': 0.001, 'epoch': 4}, {'dice_coefficient': 0.8285, 'iou_coefficient': 0.7314, 'loss': 0.4596, 'val_dice_coefficient': 0.7101, 'val_iou_coefficient': 0.5882, 'val_loss': 0.7592, 'learning_rate': 0.001, 'epoch': 5}, {'dice_coefficient': 0.83, 'iou_coefficient': 0.7343, 'loss': 0.4517, 'val_dice_coefficient': 0.8231, 'val_iou_coefficient': 0.7265, 'val_loss': 0.5249, 'learning_rate': 0.001, 'epoch': 6}, {'dice_coefficient': 0.8366, 'iou_coefficient': 0.7434, 'loss': 0.4366, 'val_dice_coefficient': 0.8043, 'val_iou_coefficient': 0.6975, 'val_loss': 0.5787, 'learning_rate': 0.001, 'epoch': 7}, {'dice_coefficient': 0.8431, 'iou_coefficient': 0.7495, 'loss': 0.4223, 'val_dice_coefficient': 0.7663, 'val_iou_coefficient': 0.6546, 'val_loss': 0.674, 'learning_rate': 0.001, 'epoch': 8}, {'dice_coefficient': 0.8475, 'iou_coefficient': 0.7557, 'loss': 0.4039, 'val_dice_coefficient': 0.8412, 'val_iou_coefficient': 0.7449, 'val_loss': 0.4815, 'learning_rate': 0.001, 'epoch': 9}, {'dice_coefficient': 0.8579, 'iou_coefficient': 0.771, 'loss': 0.3836, 'val_dice_coefficient': 0.834, 'val_iou_coefficient': 0.7421, 'val_loss': 0.4502, 'learning_rate': 0.001, 'epoch': 10}, {'dice_coefficient': 0.8587, 'iou_coefficient': 0.7701, 'loss': 0.3774, 'val_dice_coefficient': 0.8087, 'val_iou_coefficient': 0.7017, 'val_loss': 0.4945, 'learning_rate': 0.001, 'epoch': 11}, {'dice_coefficient': 0.8675, 'iou_coefficient': 0.7826, 'loss': 0.3535, 'val_dice_coefficient': 0.8561, 'val_iou_coefficient': 0.7688, 'val_loss': 0.3878, 'learning_rate': 0.001, 'epoch': 12}, {'dice_coefficient': 0.8628, 'iou_coefficient': 0.777, 'loss': 0.3681, 'val_dice_coefficient': 0.8154, 'val_iou_coefficient': 0.712, 'val_loss': 0.5091, 'learning_rate': 0.001, 'epoch': 13}, {'dice_coefficient': 0.8653, 'iou_coefficient': 0.7789, 'loss': 0.3599, 'val_dice_coefficient': 0.8361, 'val_iou_coefficient': 0.7424, 'val_loss': 0.544, 'learning_rate': 0.001, 'epoch': 14}, {'dice_coefficient': 0.873, 'iou_coefficient': 0.79, 'loss': 0.3344, 'val_dice_coefficient': 0.5969, 'val_iou_coefficient': 0.5176, 'val_loss': 1.1994, 'learning_rate': 0.001, 'epoch': 15}, {'dice_coefficient': 0.8839, 'iou_coefficient': 0.8056, 'loss': 0.3105, 'val_dice_coefficient': 0.8762, 'val_iou_coefficient': 0.7928, 'val_loss': 0.3333, 'learning_rate': 0.0005, 'epoch': 16}, {'dice_coefficient': 0.8806, 'iou_coefficient': 0.8006, 'loss': 0.3154, 'val_dice_coefficient': 0.8701, 'val_iou_coefficient': 0.7848, 'val_loss': 0.3346, 'learning_rate': 0.0005, 'epoch': 17}, {'dice_coefficient': 0.8867, 'iou_coefficient': 0.8075, 'loss': 0.3019, 'val_dice_coefficient': 0.8814, 'val_iou_coefficient': 0.7991, 'val_loss': 0.3266, 'learning_rate': 0.0005, 'epoch': 18}, {'dice_coefficient': 0.8873, 'iou_coefficient': 0.8088, 'loss': 0.3002, 'val_dice_coefficient': 0.8603, 'val_iou_coefficient': 0.7729, 'val_loss': 0.3674, 'learning_rate': 0.0005, 'epoch': 19}, {'dice_coefficient': 0.8869, 'iou_coefficient': 0.8094, 'loss': 0.3037, 'val_dice_coefficient': 0.8761, 'val_iou_coefficient': 0.7926, 'val_loss': 0.3315, 'learning_rate': 0.0005, 'epoch': 20}, {'dice_coefficient': 0.8898, 'iou_coefficient': 0.8127, 'loss': 0.2886, 'val_dice_coefficient': 0.8698, 'val_iou_coefficient': 0.784, 'val_loss': 0.3425, 'learning_rate': 0.0005, 'epoch': 21}, {'dice_coefficient': 0.896, 'iou_coefficient': 0.822, 'loss': 0.2754, 'val_dice_coefficient': 0.8838, 'val_iou_coefficient': 0.802, 'val_loss': 0.3053, 'learning_rate': 0.00025, 'epoch': 22}, {'dice_coefficient': 0.8955, 'iou_coefficient': 0.821, 'loss': 0.2699, 'val_dice_coefficient': 0.8704, 'val_iou_coefficient': 0.7885, 'val_loss': 0.3511, 'learning_rate': 0.00025, 'epoch': 23}, {'dice_coefficient': 0.898, 'iou_coefficient': 0.8256, 'loss': 0.2655, 'val_dice_coefficient': 0.8607, 'val_iou_coefficient': 0.7767, 'val_loss': 0.3765, 'learning_rate': 0.00025, 'epoch': 24}, {'dice_coefficient': 0.8982, 'iou_coefficient': 0.8255, 'loss': 0.2641, 'val_dice_coefficient': 0.8717, 'val_iou_coefficient': 0.7852, 'val_loss': 0.3247, 'learning_rate': 0.00025, 'epoch': 25}, {'dice_coefficient': 0.9021, 'iou_coefficient': 0.8311, 'loss': 0.2585, 'val_dice_coefficient': 0.8885, 'val_iou_coefficient': 0.8087, 'val_loss': 0.2874, 'learning_rate': 0.000125, 'epoch': 26}, {'dice_coefficient': 0.9018, 'iou_coefficient': 0.8308, 'loss': 0.2601, 'val_dice_coefficient': 0.8852, 'val_iou_coefficient': 0.8051, 'val_loss': 0.2968, 'learning_rate': 0.000125, 'epoch': 27}, {'dice_coefficient': 0.9021, 'iou_coefficient': 0.8307, 'loss': 0.2597, 'val_dice_coefficient': 0.8891, 'val_iou_coefficient': 0.8102, 'val_loss': 0.2877, 'learning_rate': 0.000125, 'epoch': 28}, {'dice_coefficient': 0.9019, 'iou_coefficient': 0.8306, 'loss': 0.2544, 'val_dice_coefficient': 0.8854, 'val_iou_coefficient': 0.8056, 'val_loss': 0.2926, 'learning_rate': 0.000125, 'epoch': 29}, {'dice_coefficient': 0.9022, 'iou_coefficient': 0.8316, 'loss': 0.2546, 'val_dice_coefficient': 0.8882, 'val_iou_coefficient': 0.8094, 'val_loss': 0.2862, 'learning_rate': 0.000125, 'epoch': 30}, {'dice_coefficient': 0.9008, 'iou_coefficient': 0.8295, 'loss': 0.2546, 'val_dice_coefficient': 0.8725, 'val_iou_coefficient': 0.7901, 'val_loss': 0.3192, 'learning_rate': 0.000125, 'epoch': 31}, {'dice_coefficient': 0.9053, 'iou_coefficient': 0.8361, 'loss': 0.2482, 'val_dice_coefficient': 0.8893, 'val_iou_coefficient': 0.8103, 'val_loss': 0.2872, 'learning_rate': 6.25e-05, 'epoch': 32}, {'dice_coefficient': 0.9047, 'iou_coefficient': 0.8347, 'loss': 0.2487, 'val_dice_coefficient': 0.8905, 'val_iou_coefficient': 0.8124, 'val_loss': 0.2848, 'learning_rate': 6.25e-05, 'epoch': 33}, {'dice_coefficient': 0.9046, 'iou_coefficient': 0.8349, 'loss': 0.2449, 'val_dice_coefficient': 0.888, 'val_iou_coefficient': 0.8094, 'val_loss': 0.2853, 'learning_rate': 6.25e-05, 'epoch': 34}, {'dice_coefficient': 0.9045, 'iou_coefficient': 0.8349, 'loss': 0.2467, 'val_dice_coefficient': 0.8909, 'val_iou_coefficient': 0.8132, 'val_loss': 0.2816, 'learning_rate': 6.25e-05, 'epoch': 35}], 'classifier_head': [{'accuracy': 0.6611, 'auc': 0.6787, 'loss': 0.6581, 'val_accuracy': 0.5944, 'val_auc': 0.7157, 'val_loss': 0.6388, 'val_specificity_at_target': 0.4621, 'epoch': 1}, {'accuracy': 0.7097, 'auc': 0.8146, 'loss': 0.5219, 'val_accuracy': 0.6278, 'val_auc': 0.7405, 'val_loss': 0.6391, 'val_specificity_at_target': 0.5724, 'epoch': 2}, {'accuracy': 0.7139, 'auc': 0.8285, 'loss': 0.5054, 'val_accuracy': 0.7722, 'val_auc': 0.7558, 'val_loss': 0.5191, 'val_specificity_at_target': 0.5655, 'epoch': 3}, {'accuracy': 0.7292, 'auc': 0.82, 'loss': 0.5053, 'val_accuracy': 0.7222, 'val_auc': 0.7486, 'val_loss': 0.5361, 'val_specificity_at_target': 0.5724, 'epoch': 4}, {'accuracy': 0.7875, 'auc': 0.8802, 'loss': 0.4371, 'val_accuracy': 0.7722, 'val_auc': 0.7652, 'val_loss': 0.4493, 'val_specificity_at_target': 0.5448, 'epoch': 5}, {'accuracy': 0.7681, 'auc': 0.8753, 'loss': 0.4331, 'val_accuracy': 0.6889, 'val_auc': 0.7657, 'val_loss': 0.5522, 'val_specificity_at_target': 0.5931, 'epoch': 6}, {'accuracy': 0.7917, 'auc': 0.8843, 'loss': 0.4222, 'val_accuracy': 0.7389, 'val_auc': 0.7518, 'val_loss': 0.4921, 'val_specificity_at_target': 0.5172, 'epoch': 7}, {'accuracy': 0.8139, 'auc': 0.9107, 'loss': 0.3817, 'val_accuracy': 0.6111, 'val_auc': 0.7544, 'val_loss': 0.6953, 'val_specificity_at_target': 0.5517, 'epoch': 8}, {'accuracy': 0.8153, 'auc': 0.9126, 'loss': 0.374, 'val_accuracy': 0.7389, 'val_auc': 0.7663, 'val_loss': 0.5325, 'val_specificity_at_target': 0.6, 'epoch': 9}, {'accuracy': 0.8458, 'auc': 0.9247, 'loss': 0.3537, 'val_accuracy': 0.7389, 'val_auc': 0.746, 'val_loss': 0.5107, 'val_specificity_at_target': 0.5931, 'epoch': 10}, {'accuracy': 0.8361, 'auc': 0.9327, 'loss': 0.3343, 'val_accuracy': 0.6889, 'val_auc': 0.7532, 'val_loss': 0.6088, 'val_specificity_at_target': 0.6, 'epoch': 11}, {'accuracy': 0.8444, 'auc': 0.926, 'loss': 0.3439, 'val_accuracy': 0.7111, 'val_auc': 0.7594, 'val_loss': 0.5396, 'val_specificity_at_target': 0.6, 'epoch': 12}, {'accuracy': 0.8722, 'auc': 0.9473, 'loss': 0.3077, 'val_accuracy': 0.6944, 'val_auc': 0.7542, 'val_loss': 0.6118, 'val_specificity_at_target': 0.5586, 'epoch': 13}, {'accuracy': 0.8694, 'auc': 0.9541, 'loss': 0.2847, 'val_accuracy': 0.7611, 'val_auc': 0.7548, 'val_loss': 0.4896, 'val_specificity_at_target': 0.5586, 'epoch': 14}], 'classifier_fine': [{'accuracy': 0.8569, 'auc': 0.9389, 'loss': 0.327, 'val_accuracy': 0.7444, 'val_auc': 0.7626, 'val_loss': 0.5111, 'val_specificity_at_target': 0.5517, 'epoch': 1}, {'accuracy': 0.8542, 'auc': 0.9347, 'loss': 0.3348, 'val_accuracy': 0.6944, 'val_auc': 0.7634, 'val_loss': 0.5696, 'val_specificity_at_target': 0.5931, 'epoch': 2}, {'accuracy': 0.85, 'auc': 0.938, 'loss': 0.3292, 'val_accuracy': 0.75, 'val_auc': 0.7635, 'val_loss': 0.5109, 'val_specificity_at_target': 0.5586, 'epoch': 3}, {'accuracy': 0.8653, 'auc': 0.9428, 'loss': 0.3142, 'val_accuracy': 0.7611, 'val_auc': 0.764, 'val_loss': 0.5039, 'val_specificity_at_target': 0.531, 'epoch': 4}, {'accuracy': 0.8639, 'auc': 0.9524, 'loss': 0.2927, 'val_accuracy': 0.7444, 'val_auc': 0.7622, 'val_loss': 0.5156, 'val_specificity_at_target': 0.531, 'epoch': 5}, {'accuracy': 0.8722, 'auc': 0.9483, 'loss': 0.3008, 'val_accuracy': 0.75, 'val_auc': 0.7624, 'val_loss': 0.5138, 'val_specificity_at_target': 0.5448, 'epoch': 6}], 'source_run': '/content/drive/MyDrive/Melanoma_ISIC2016_Final_Candidate/20261007T163637_386595Z', 'note': 'Rounded console summaries recovered from uploaded completed notebooks. Training display values can differ from Keras History/CSVLogger epoch values. Segmentation source: Specificity_Experiment(3), 35 epochs, matching validation Dice/IoU; checkpoint identity cannot be proved from logs alone. Classifier source: Final_Candidate(1), head 14 epochs and fine-tuning 6 epochs. These are historical curves, not a new training run.'}


## 3. Mode, saved candidate and output folder
Keep evaluate_saved to retain your frozen candidate. Change SAVED_RUN_DIR only if the completed run was moved.

In [ ]:
SEED = 42
keras.utils.set_random_seed(SEED)
SEG_IMAGE_SIZE = (256, 256)
CLS_IMAGE_SIZE = (224, 224)
SEG_BATCH_SIZE = 4
CLS_BATCH_SIZE = 8
SEG_EPOCHS = 35
CLS_HEAD_EPOCHS = 15
CLS_FINE_TUNE_EPOCHS = 8
SEG_LEARNING_RATE = 1e-3
CLS_HEAD_LR = 3e-4
CLS_FINE_TUNE_LR = 1e-5
SEG_THRESHOLD = 0.5
TARGET_VALIDATION_SENSITIVITY = 0.80
SAVE_RESULTS_TO_DRIVE = True

RUNTIME_ROOT = Path('/content/melanoma_attention_reporting')
DOWNLOAD_DIR = RUNTIME_ROOT / 'downloads'
DATA_DIR = RUNTIME_ROOT / 'images'
ROI_DIR = RUNTIME_ROOT / 'predicted_roi'
PRED_MASK_DIR = RUNTIME_ROOT / 'predicted_masks'
for folder in (DOWNLOAD_DIR, DATA_DIR, ROI_DIR, PRED_MASK_DIR):
    folder.mkdir(parents=True, exist_ok=True)
if SAVE_RESULTS_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT_DIR = Path('/content/drive/MyDrive/Melanoma_Attention_UNet_Reporting_and_App')
    except Exception as exc:
        print('Drive unavailable; writing output in /content:', exc)
        OUTPUT_DIR = RUNTIME_ROOT / 'results'
else:
    OUTPUT_DIR = RUNTIME_ROOT / 'results'
# Each execution gets its own folder, preserving earlier checkpoints.
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT_DIR = OUTPUT_DIR / RUN_ID
MODELS_DIR = OUTPUT_DIR / 'models'
RESULTS_DIR = OUTPUT_DIR / 'tables'
for folder in (MODELS_DIR, RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print('Results:', OUTPUT_DIR)

# DEFAULT: evaluate the existing frozen candidate. This does not train new models.
MODE = "evaluate_saved"  # Alternative: "train_new" creates a separate new experiment.
SAVED_RUN_DIR = Path('/content/drive/MyDrive/Melanoma_ISIC2016_Final_Candidate/20261007T163637_386595Z')
SHARE_APP = True  # Colab demo link; keep the Colab runtime connected.
if MODE not in ("evaluate_saved", "train_new"):
    raise ValueError("MODE must be 'evaluate_saved' or 'train_new'.")
REUSE_SAVED_SEGMENTER = MODE == "evaluate_saved"
PREVIOUS_SEGMENTER = SAVED_RUN_DIR / "models" / "attention_unet_best.keras"
FIGURES_DIR = OUTPUT_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)
if MODE == "evaluate_saved":
    needed = [PREVIOUS_SEGMENTER,
              SAVED_RUN_DIR / "models" / "selected_final_classifier.keras",
              SAVED_RUN_DIR / "tables" / "selected_configuration.json"]
    missing = [str(p) for p in needed if not p.is_file()]
    if missing:
        raise FileNotFoundError("Your completed run was not found. Update SAVED_RUN_DIR. Missing:\n"
                                + "\n".join(missing))
    SAVED_CONFIGURATION = json.loads(needed[-1].read_text())
    for old_path in (SAVED_RUN_DIR / "tables").iterdir():
        if old_path.is_file() and old_path.suffix in (".json", ".csv"):
            shutil.copy2(old_path, RESULTS_DIR / old_path.name)
    shutil.copy2(needed[1], MODELS_DIR / "selected_final_classifier.keras")
print("Mode:", MODE)

# Confidence intervals and optional comparisons (used automatically in Sections 18–23).
BOOTSTRAP_REPLICATES = 5000
RANDOM_SEED = 20261107
# Keep None to skip an experiment, or enter its saved reporting run folder.
PRIOR_RUN_DIRS = {
    'hair_removal_on': None,
    'hair_removal_off': None,
    'augmentation_variant': None,
    'ROI_or_classifier_variant': None,
    'other_segmentation_pipeline': None,
}


## 4. Download the official ISIC 2016 Task 3B files

This may download about 850 MB. If Colab disconnects, downloaded runtime files
may be lost. The official test labels are held out from all training, model
selection, and threshold selection.

In [ ]:
BASE_URL = 'https://isic-archive.s3.amazonaws.com/challenges/2016/'
files = {
    'training_zip': 'ISBI2016_ISIC_Part3B_Training_Data.zip',
    'training_csv': 'ISBI2016_ISIC_Part3B_Training_GroundTruth.csv',
    'test_zip': 'ISBI2016_ISIC_Part3B_Test_Data.zip',
    'test_csv': 'ISBI2016_ISIC_Part3B_Test_GroundTruth.csv',
}

def download(url, destination):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 100:
        print('Already downloaded:', destination.name)
        return destination
    temporary = destination.with_suffix(destination.suffix + '.part')
    start = temporary.stat().st_size if temporary.exists() else 0
    headers = {'Range': f'bytes={start}-'} if start else {}
    print('Downloading:', destination.name)
    with requests.get(url, stream=True, headers=headers, timeout=(30, 120)) as response:
        response.raise_for_status()
        if start and response.status_code != 206:
            start = 0
            temporary.unlink(missing_ok=True)
        with open(temporary, 'ab' if start else 'wb') as handle:
            for part in response.iter_content(chunk_size=8 * 1024 * 1024):
                if part:
                    handle.write(part)
    temporary.replace(destination)
    return destination

paths = {key: download(BASE_URL + filename, DOWNLOAD_DIR / filename)
         for key, filename in files.items()}
for key in ('training_zip', 'test_zip'):
    assert zipfile.is_zipfile(paths[key]), f'Invalid ZIP: {paths[key]}'
print('Official files downloaded.')


## 5. Reproduce and verify the original training/validation split

In [ ]:
def read_labels(path, expected):
    # The 2016 training CSV uses benign/malignant, while the official test
    # CSV uses numeric 0/1. Per ISIC, 0 = benign and 1 = malignant.
    frame = pd.read_csv(path, header=None, names=['image_id', 'raw_label'],
                        dtype=str)
    frame['image_id'] = frame['image_id'].astype(str).str.strip()
    normalized = frame['raw_label'].astype(str).str.strip().str.lower()
    frame['label'] = normalized.map({
        'benign': 0, 'malignant': 1, '0': 0, '0.0': 0, '1': 1, '1.0': 1,
    })
    frame = frame[frame['image_id'].str.fullmatch(r'ISIC_\d{7}') &
                  frame['label'].notna()].copy()
    assert len(frame) == expected and frame['image_id'].is_unique, (
        f'Expected {expected} unique labels in {path}, got {len(frame)}. '
        'Check CSV columns and label values.'
    )
    frame['label'] = frame['label'].astype(int)
    frame['diagnosis'] = np.where(frame['label'] == 1, 'malignant', 'benign')
    return frame[['image_id', 'diagnosis', 'label']].reset_index(drop=True)

training_labels = read_labels(paths['training_csv'], 900)
official_test_labels = read_labels(paths['test_csv'], 379)
assert set(training_labels.image_id).isdisjoint(set(official_test_labels.image_id))
train_df, val_df = train_test_split(
    training_labels, test_size=0.20, stratify=training_labels['label'],
    random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = official_test_labels.copy()
for name, frame in [('train', train_df), ('validation', val_df), ('official_test', test_df)]:
    print(name, 'total:', len(frame), 'melanoma:', int(frame['label'].sum()))
    frame[['image_id', 'diagnosis', 'label']].to_csv(
        RESULTS_DIR / f'{name}_image_ids.csv', index=False)

split_hash = hashlib.sha256(('|'.join(sorted(train_df.image_id)) + '\n' +
                             '|'.join(sorted(val_df.image_id))).encode()).hexdigest()
if MODE == "evaluate_saved" and SAVED_CONFIGURATION.get("split_sha256") != split_hash:
    raise ValueError("This split does not match the frozen candidate. Do not change SEED or split settings.")


## 6. Extract matched images and expert masks

In [ ]:
def extract_pairs(zip_path, labels, split_name):
    destination = DATA_DIR / split_name
    destination.mkdir(parents=True, exist_ok=True)
    records = []
    with zipfile.ZipFile(zip_path) as archive:
        member_map = {Path(name).name: name for name in archive.namelist()
                      if not name.endswith('/')}
        for image_id in labels['image_id']:
            image_name = f'{image_id}.jpg'
            mask_name = f'{image_id}_Segmentation.png'
            assert image_name in member_map and mask_name in member_map, image_id
            image_path, mask_path = destination / image_name, destination / mask_name
            for name, out in [(image_name, image_path), (mask_name, mask_path)]:
                if not out.exists():
                    with archive.open(member_map[name]) as src, open(out, 'wb') as dst:
                        shutil.copyfileobj(src, dst)
            records.append({'image_id': image_id, 'image_path': str(image_path),
                            'mask_path': str(mask_path)})
    return pd.DataFrame(records)

training_paths = extract_pairs(paths['training_zip'], training_labels, 'development')
test_paths = extract_pairs(paths['test_zip'], test_df, 'official_test')
train_df = train_df.merge(training_paths, on='image_id', validate='one_to_one')
val_df = val_df.merge(training_paths, on='image_id', validate='one_to_one')
test_df = test_df.merge(test_paths, on='image_id', validate='one_to_one')
for key in ('training_zip', 'test_zip'):
    paths[key].unlink(missing_ok=True)
print('Extracted:', len(train_df), len(val_df), len(test_df))


## 7. Visual check: original image and expert lesion mask

In [ ]:
def load_rgb(path):
    with Image.open(path) as picture:
        return picture.convert('RGB').copy()

def load_mask(path):
    with Image.open(path) as picture:
        return picture.convert('L').copy()

def segmentation_overlay(image_pil, mask_binary, alpha=0.35):
    image = np.asarray(image_pil.convert('RGB'), dtype=np.float32).copy()
    mask = Image.fromarray(np.uint8(np.asarray(mask_binary) > 0) * 255)
    mask = np.asarray(mask.resize(image_pil.size, Image.Resampling.NEAREST)) > 0
    image[mask] = (1-alpha) * image[mask] + alpha * np.array([255, 0, 0])
    return Image.fromarray(np.uint8(np.clip(image, 0, 255)))

examples = pd.concat([train_df.query('label == 0').head(2),
                      train_df.query('label == 1').head(2)])
fig, axes = plt.subplots(len(examples), 3, figsize=(12, 3.4*len(examples)))
for i, (_, row) in enumerate(examples.iterrows()):
    image = load_rgb(row['image_path'])
    mask = np.asarray(load_mask(row['mask_path'])) > 127
    for j, (panel, title) in enumerate(zip(
        [image, mask, segmentation_overlay(image, mask)],
        [f"{row['image_id']} {row['diagnosis']}", 'Expert mask', 'Overlay'])):
        axes[i, j].imshow(panel, cmap='gray' if j == 1 else None)
        axes[i, j].set_title(title)
        axes[i, j].axis('off')
plt.tight_layout(); plt.show()


## 8. Segmentation data and loss
Test data are not augmented. Loss is BCE + soft Dice loss; reported mask Dice/IoU use thresholded predictions.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def decode_segmentation_image(image_path):
    image = tf.io.decode_jpeg(tf.io.read_file(image_path), channels=3)
    return tf.cast(tf.image.resize(image, SEG_IMAGE_SIZE, method='bilinear'),
                   tf.float32) / 255.0

def decode_segmentation_mask(mask_path):
    mask = tf.io.decode_png(tf.io.read_file(mask_path), channels=1)
    mask = tf.image.resize(mask, SEG_IMAGE_SIZE, method='nearest')
    return tf.cast(mask > 127, tf.float32)

def decode_segmentation_pair(image_path, mask_path):
    return decode_segmentation_image(image_path), decode_segmentation_mask(mask_path)

def paired_segmentation_augmentation(image, mask):
    seed1 = tf.random.uniform([2], maxval=2**31-1, dtype=tf.int32)
    seed2 = tf.random.uniform([2], maxval=2**31-1, dtype=tf.int32)
    image = tf.image.stateless_random_flip_left_right(image, seed1)
    mask = tf.image.stateless_random_flip_left_right(mask, seed1)
    image = tf.image.stateless_random_flip_up_down(image, seed2)
    mask = tf.image.stateless_random_flip_up_down(mask, seed2)
    turns = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
    image, mask = tf.image.rot90(image, turns), tf.image.rot90(mask, turns)
    image = tf.image.random_brightness(image, max_delta=0.06)
    image = tf.image.random_contrast(image, lower=0.90, upper=1.10)
    return tf.clip_by_value(image, 0.0, 1.0), mask

def make_seg_dataset(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((
        frame['image_path'].astype(str).values,
        frame['mask_path'].astype(str).values))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_segmentation_pair, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(paired_segmentation_augmentation, num_parallel_calls=AUTOTUNE)
    return ds.batch(SEG_BATCH_SIZE).prefetch(AUTOTUNE)

train_seg_ds = make_seg_dataset(train_df, training=True)
val_seg_ds = make_seg_dataset(val_df)
test_seg_ds = make_seg_dataset(test_df)
images_batch, masks_batch = next(iter(train_seg_ds))
print('Image and mask shapes:', images_batch.shape, masks_batch.shape)
print('Image range:', float(tf.reduce_min(images_batch)),
      'to', float(tf.reduce_max(images_batch)))
print('Mask values:', np.unique(masks_batch.numpy()))


In [ ]:
EPS = 1e-7
SPATIAL_AXES = (1, 2, 3)

@keras.utils.register_keras_serializable(package='MelanomaAI')
def soft_dice_coefficient(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=SPATIAL_AXES)
    denominator = tf.reduce_sum(y_true + y_pred, axis=SPATIAL_AXES)
    return tf.reduce_mean((2.0 * intersection + EPS) / (denominator + EPS))

@keras.utils.register_keras_serializable(package='MelanomaAI')
def dice_coefficient(y_true, y_pred):
    # Hard per-image Dice averaged across the current batch.
    y_true = tf.cast(y_true > 0.5, tf.float32)
    y_pred = tf.cast(y_pred >= SEG_THRESHOLD, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=SPATIAL_AXES)
    denominator = tf.reduce_sum(y_true + y_pred, axis=SPATIAL_AXES)
    return tf.reduce_mean((2.0 * intersection + EPS) / (denominator + EPS))

@keras.utils.register_keras_serializable(package='MelanomaAI')
def dice_loss(y_true, y_pred):
    return 1.0 - soft_dice_coefficient(y_true, y_pred)

@keras.utils.register_keras_serializable(package='MelanomaAI')
def iou_coefficient(y_true, y_pred):
    y_true = tf.cast(y_true > 0.5, tf.float32)
    y_pred = tf.cast(y_pred >= SEG_THRESHOLD, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=SPATIAL_AXES)
    union = tf.reduce_sum(y_true + y_pred - y_true * y_pred, axis=SPATIAL_AXES)
    return tf.reduce_mean((intersection + EPS) / (union + EPS))

@keras.utils.register_keras_serializable(package='MelanomaAI')
def combined_segmentation_loss(y_true, y_pred):
    bce = tf.reduce_mean(keras.losses.binary_crossentropy(y_true, y_pred))
    return bce + dice_loss(y_true, y_pred)


## 9. Attention U-Net architecture and saved model
Training and mask inference selection run only in train_new mode.

In [ ]:
def conv_block(
    x,
    filters,
    name,
):
    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        use_bias=False,
        name=f"{name}_conv1",
    )(x)

    x = layers.BatchNormalization(
        name=f"{name}_bn1"
    )(x)

    x = layers.Activation(
        "relu",
        name=f"{name}_relu1",
    )(x)

    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        use_bias=False,
        name=f"{name}_conv2",
    )(x)

    x = layers.BatchNormalization(
        name=f"{name}_bn2"
    )(x)

    x = layers.Activation(
        "relu",
        name=f"{name}_relu2",
    )(x)

    return x


def attention_gate(
    skip,
    gating,
    inter_filters,
    name,
):
    theta = layers.Conv2D(
        inter_filters,
        1,
        padding="same",
        name=f"{name}_theta",
    )(skip)

    phi = layers.Conv2D(
        inter_filters,
        1,
        padding="same",
        name=f"{name}_phi",
    )(gating)

    x = layers.Add(
        name=f"{name}_add"
    )([theta, phi])

    x = layers.Activation(
        "relu",
        name=f"{name}_relu",
    )(x)

    psi = layers.Conv2D(
        1,
        1,
        padding="same",
        activation="sigmoid",
        name=f"{name}_psi",
    )(x)

    return layers.Multiply(
        name=f"{name}_weighted_skip"
    )([skip, psi])


def build_attention_unet(
    input_shape=(256, 256, 3),
):
    inputs = keras.Input(
        shape=input_shape,
        name="image",
    )

    c1 = conv_block(inputs, 32, "enc1")
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 64, "enc2")
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 128, "enc3")
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 256, "enc4")
    p4 = layers.MaxPooling2D()(c4)

    bottleneck = conv_block(
        p4,
        512,
        "bottleneck",
    )

    def decoder_step(
        x,
        skip,
        filters,
        level,
    ):
        x = layers.Conv2DTranspose(
            filters,
            2,
            strides=2,
            padding="same",
            name=f"up{level}",
        )(x)

        skip = attention_gate(
            skip,
            x,
            max(filters // 2, 1),
            name=f"attention{level}",
        )

        x = layers.Concatenate(
            name=f"concat{level}"
        )([x, skip])

        return conv_block(
            x,
            filters,
            f"dec{level}",
        )

    d4 = decoder_step(bottleneck, c4, 256, 4)
    d3 = decoder_step(d4, c3, 128, 3)
    d2 = decoder_step(d3, c2, 64, 2)
    d1 = decoder_step(d2, c1, 32, 1)

    outputs = layers.Conv2D(
        1,
        1,
        activation="sigmoid",
        dtype="float32",
        name="segmentation_mask",
    )(d1)

    return keras.Model(
        inputs,
        outputs,
        name="Attention_U_Net",
    )


In [ ]:
def compile_segmentation_model(model):
    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=SEG_LEARNING_RATE
        ),
        loss=combined_segmentation_loss,
        metrics=[
            dice_coefficient,
            iou_coefficient,
        ],
    )

def segmentation_callbacks(model_path):
    return [
        keras.callbacks.CSVLogger(str(RESULTS_DIR / "segmentation_training_history.csv")),
        keras.callbacks.ModelCheckpoint(
            model_path,
            monitor="val_dice_coefficient",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_dice_coefficient",
            mode="max",
            patience=8,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_dice_coefficient",
            mode="max",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

def predict_segmentation(images, four_view=False, model=None):
    model = selected_segmenter if model is None else model
    images = np.asarray(images, dtype=np.float32)
    if not four_view:
        return model.predict(images, batch_size=SEG_BATCH_SIZE, verbose=0)[..., 0]
    predictions = []
    for axes in [(), (2,), (1,), (1, 2)]:
        transformed = np.flip(images, axis=axes) if axes else images
        scores = model.predict(np.ascontiguousarray(transformed),
                               batch_size=SEG_BATCH_SIZE, verbose=0)[..., 0]
        # Undo each flip before averaging the spatial probability maps.
        scores = np.flip(scores, axis=axes) if axes else scores
        predictions.append(scores)
    return np.mean(np.stack(predictions), axis=0)

def mask_metrics(truth, probabilities, threshold):
    prediction = probabilities >= threshold
    intersection = np.logical_and(truth, prediction).sum(axis=(1, 2))
    total = truth.sum(axis=(1, 2)) + prediction.sum(axis=(1, 2))
    union = np.logical_or(truth, prediction).sum(axis=(1, 2))
    return (2*intersection + 1e-7)/(total + 1e-7), (intersection + 1e-7)/(union + 1e-7)

ATTENTION_MODEL_PATH = MODELS_DIR / 'attention_unet_best.keras'
if MODE == "evaluate_saved":
    shutil.copy2(PREVIOUS_SEGMENTER, ATTENTION_MODEL_PATH)
    attention_training_seconds = 0.0
    attention_history = {}
    SEGMENTER_REUSED = True
else:
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    attention_model = build_attention_unet(input_shape=(*SEG_IMAGE_SIZE, 3))
    compile_segmentation_model(attention_model)
    start_time = time.time()
    attention_fit = attention_model.fit(
        train_seg_ds, validation_data=val_seg_ds, epochs=SEG_EPOCHS,
        callbacks=segmentation_callbacks(ATTENTION_MODEL_PATH), verbose=1)
    attention_training_seconds = time.time() - start_time
    attention_history = attention_fit.history
    (RESULTS_DIR / "segmentation_training_history.json").write_text(
        json.dumps({k: [float(v) for v in values] for k, values in attention_history.items()}, indent=2))
    SEGMENTER_REUSED = False
    del attention_model, attention_fit
    gc.collect()

selected_segmenter = keras.models.load_model(ATTENTION_MODEL_PATH, compile=False)
compile_segmentation_model(selected_segmenter)
attention_validation = selected_segmenter.evaluate(val_seg_ds, verbose=0, return_dict=True)
print("Single-view validation loss / Dice / IoU:", attention_validation)
print("Segmenter parameters:", selected_segmenter.count_params())
SELECTED_SEGMENTER_NAME = 'Attention U-Net'
SELECTED_SEGMENTER_PATH = str(ATTENTION_MODEL_PATH)

if MODE == "train_new":
    segmentation_selection_rows = []
    for method, use_flips in [('single', False), ('4-flip average', True)]:
        dice_parts = {t: [] for t in (0.4, 0.5, 0.6)}
        iou_parts = {t: [] for t in (0.4, 0.5, 0.6)}
        for start in range(0, len(val_df), 12):
            rows = val_df.iloc[start:start+12]
            images = np.stack([decode_segmentation_image(p).numpy()
                               for p in rows.image_path])
            truth = np.stack([decode_segmentation_mask(p).numpy()[..., 0] > 0.5
                              for p in rows.mask_path])
            scores = predict_segmentation(images, four_view=use_flips)
            for threshold in dice_parts:
                dice, iou = mask_metrics(truth, scores, threshold)
                dice_parts[threshold].extend(dice.tolist())
                iou_parts[threshold].extend(iou.tolist())
        for threshold in dice_parts:
            segmentation_selection_rows.append({
                'Method': method, 'Threshold': threshold,
                'Validation Dice': float(np.mean(dice_parts[threshold])),
                'Validation IoU': float(np.mean(iou_parts[threshold])),
                'Worst validation Dice': float(np.min(dice_parts[threshold])),
            })

    segmentation_selection = pd.DataFrame(segmentation_selection_rows).sort_values(
        ['Validation Dice', 'Validation IoU'], ascending=False).reset_index(drop=True)
    SELECTED_SEGMENTATION_METHOD = str(segmentation_selection.loc[0, 'Method'])
    SEG_THRESHOLD = float(segmentation_selection.loc[0, 'Threshold'])
    display(segmentation_selection.round(4))
    print('Selected mask inference:', SELECTED_SEGMENTATION_METHOD, SEG_THRESHOLD)
    segmentation_selection.to_csv(RESULTS_DIR / 'validation_segmentation_inference.csv',
                                  index=False)
else:
    SELECTED_SEGMENTATION_METHOD = SAVED_CONFIGURATION['segmentation_method']
    SEG_THRESHOLD = float(SAVED_CONFIGURATION['segmentation_threshold'])
    selection_path = RESULTS_DIR / 'validation_segmentation_inference.csv'
    segmentation_selection = pd.read_csv(selection_path) if selection_path.is_file() else pd.DataFrame()
    if len(segmentation_selection):
        display(segmentation_selection.round(4))
    print('Keeping frozen mask inference:', SELECTED_SEGMENTATION_METHOD, SEG_THRESHOLD)
checkpoint_hash = hashlib.sha256(ATTENTION_MODEL_PATH.read_bytes()).hexdigest()
if MODE == 'evaluate_saved':
    provenance = SAVED_RUN_DIR / 'tables' / 'segmentation_configuration.json'
    if provenance.is_file():
        expected = json.loads(provenance.read_text()).get('checkpoint_sha256')
        if expected and expected != checkpoint_hash:
            raise ValueError('Saved segmentation checkpoint differs from its configuration.')
(RESULTS_DIR / 'segmentation_configuration.json').write_text(json.dumps({
    'method': SELECTED_SEGMENTATION_METHOD, 'threshold': SEG_THRESHOLD,
    'checkpoint_sha256': checkpoint_hash, 'split_sha256': split_hash,
    'checkpoint_reused': SEGMENTER_REUSED,
}, indent=2))


## 10. Predicted masks and lesion ROIs
Largest connected component, 15% padding, square edge padding and quality-95 JPEG encoding match the frozen classifier.

In [ ]:
from scipy import ndimage

def crop_from_prediction(image_pil, predicted_mask, padding=0.15):
    image = image_pil.convert('RGB')
    width, height = image.size
    binary = Image.fromarray(np.uint8(np.asarray(predicted_mask) > 0) * 255)
    binary = np.asarray(binary.resize(image.size, Image.Resampling.NEAREST)) > 0
    labelled, count = ndimage.label(binary)
    reason = 'cropped'
    if count == 0:
        crop, reason = image, 'empty predicted mask'
    else:
        sizes = np.bincount(labelled.ravel())
        sizes[0] = 0
        lesion = labelled == int(sizes.argmax())
        area = int(lesion.sum())
        if area < max(64, int(width*height*0.001)) or area > width*height*0.80:
            crop, reason = image, 'implausible predicted area'
        else:
            ys, xs = np.where(lesion)
            span = max(np.ptp(xs)+1, np.ptp(ys)+1)
            margin = int(np.ceil(span*padding))
            x0, x1 = max(0, int(xs.min())-margin), min(width, int(xs.max())+margin+1)
            y0, y1 = max(0, int(ys.min())-margin), min(height, int(ys.max())+margin+1)
            crop = image.crop((x0, y0, x1, y1))
    a = np.asarray(crop)
    h, w = a.shape[:2]
    side = max(h, w)
    top, left = (side-h)//2, (side-w)//2
    square = np.pad(a, ((top, side-h-top), (left, side-w-left), (0, 0)), mode='edge')
    roi = Image.fromarray(square).resize(
        (CLS_IMAGE_SIZE[1], CLS_IMAGE_SIZE[0]), Image.Resampling.BILINEAR)
    return roi, reason

def make_predicted_inputs(frame, split_name):
    records = []
    for begin in range(0, len(frame), 12):
        rows = list(frame.iloc[begin:begin+12].itertuples(index=False))
        originals = [load_rgb(row.image_path) for row in rows]
        images = np.stack([
            decode_segmentation_image(row.image_path).numpy() for row in rows])
        probabilities = predict_segmentation(
            images, four_view=(SELECTED_SEGMENTATION_METHOD == '4-flip average'))
        for row, original, probability in zip(rows, originals, probabilities):
            binary = probability >= SEG_THRESHOLD
            roi, reason = crop_from_prediction(original, binary)
            mask_path = PRED_MASK_DIR / f'{row.image_id}_pred.png'
            roi_path = ROI_DIR / f'{row.image_id}_roi.jpg'
            Image.fromarray(np.uint8(binary)*255).save(mask_path)
            roi.save(roi_path, quality=95)
            records.append({'image_id': row.image_id, 'roi_path': str(roi_path),
                            'pred_mask_path': str(mask_path), 'roi_reason': reason})
        if (begin + len(rows)) % 120 == 0 or begin + len(rows) == len(frame):
            print(split_name, begin + len(rows), '/', len(frame))
    return frame.drop(columns=['roi_path', 'pred_mask_path', 'roi_reason'],
                      errors='ignore').merge(
        pd.DataFrame(records), on='image_id', validate='one_to_one')

if MODE == 'train_new':
    train_df = make_predicted_inputs(train_df, 'train')
val_df = make_predicted_inputs(val_df, 'validation')
test_df = make_predicted_inputs(test_df, 'official test')
for name, frame in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    if 'roi_reason' in frame:
        print(name, 'ROI reasons:', frame['roi_reason'].value_counts().to_dict())

examples = val_df.head(4)
fig, axes = plt.subplots(4, 4, figsize=(13, 12))
for i, (_, row) in enumerate(examples.iterrows()):
    original = load_rgb(row['image_path'])
    pred_mask = np.asarray(load_mask(row['pred_mask_path'])) > 127
    expert_mask = np.asarray(load_mask(row['mask_path'])) > 127
    panels = [original, segmentation_overlay(original, expert_mask),
              segmentation_overlay(original, pred_mask), load_rgb(row['roi_path'])]
    for j, (panel, title) in enumerate(zip(panels, ['Original', 'Expert mask',
                                                    'Predicted mask', 'Predicted ROI'])):
        axes[i, j].imshow(panel)
        axes[i, j].set_title(f"{row['image_id']} — {title}" if j == 0 else title)
        axes[i, j].axis('off')
plt.tight_layout(); plt.show()


## 11. Classifier datasets

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def decode_classifier_image(path):
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.cast(tf.image.resize(image, CLS_IMAGE_SIZE), tf.float32)

def classifier_dataset(frame, view, training=False):
    ds = tf.data.Dataset.from_tensor_slices((
        frame['image_path'].astype(str).to_numpy(),
        frame['roi_path'].astype(str).to_numpy(),
        frame['label'].astype(np.float32).to_numpy(),
    ))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)

    def convert(full_path, roi_path, label):
        full = decode_classifier_image(full_path)
        if view == 'dual':
            roi = decode_classifier_image(roi_path)
            return {'full_image': full, 'roi_image': roi}, label
        return full, label

    return ds.map(convert, num_parallel_calls=AUTOTUNE).batch(CLS_BATCH_SIZE).prefetch(AUTOTUNE)

from sklearn.utils.class_weight import compute_class_weight
train_labels = train_df['label'].to_numpy()
values = np.unique(train_labels)
weights = compute_class_weight('balanced', classes=values, y=train_labels)
CLASS_WEIGHTS = {int(k): float(v) for k, v in zip(values, weights)}
print('Training class weights:', CLASS_WEIGHTS)


## 12. Full-image + ROI EfficientNetB0

In [ ]:
def augmentation(name, seed):
    return keras.Sequential([
        layers.RandomFlip('horizontal_and_vertical', seed=seed),
        layers.RandomRotation(0.06, seed=seed+1),
        layers.RandomContrast(0.12, seed=seed+2),
        layers.RandomBrightness(0.06, value_range=(0, 255), seed=seed+3),
    ], name=name)

def build_classifier(view, pretrained_weights='imagenet'):
    backbone = keras.applications.EfficientNetB0(
        include_top=False, weights=pretrained_weights, input_shape=(*CLS_IMAGE_SIZE, 3),
        name='efficientnetb0')
    backbone.trainable = False
    gap = layers.GlobalAveragePooling2D(name='gap')
    full_input = keras.Input(shape=(*CLS_IMAGE_SIZE, 3), name='full_image')
    full = augmentation('full_augmentation', SEED)(full_input)
    full_features = gap(backbone(full, training=False))
    if view == 'dual':
        roi_input = keras.Input(shape=(*CLS_IMAGE_SIZE, 3), name='roi_image')
        roi = augmentation('roi_augmentation', SEED+17)(roi_input)
        roi_features = gap(backbone(roi, training=False))
        features = layers.Concatenate(name='combine_features')(
            [full_features, roi_features])
        inputs = [full_input, roi_input]
    else:
        features = full_features
        inputs = full_input
    x = layers.Dense(128, activation='relu', name='hidden_dense')(features)
    x = layers.Dropout(0.35, name='head_dropout')(x)
    result = layers.Dense(1, activation='sigmoid', name='prediction')(x)
    return keras.Model(inputs, result, name=f'melanoma_{view}'), backbone

def compile_classifier(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[keras.metrics.AUC(name='auc'),
                 keras.metrics.BinaryAccuracy(name='accuracy')],
    )


## 13. Classifier training (skipped for the frozen candidate)
Exact epoch CSVs are written during each new training phase.

In [ ]:
def classification_counts(y_true, probability, threshold):
    probability = np.asarray(probability).reshape(-1)
    y_true = np.asarray(y_true, dtype=int).reshape(-1)
    predicted = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    sens = float(tp / (tp + fn))
    spec = float(tn / (tn + fp))
    return {'Threshold': float(threshold), 'TP': int(tp), 'TN': int(tn),
            'FP': int(fp), 'FN': int(fn), 'Sensitivity': sens,
            'Specificity': spec, 'Accuracy': float((tp + tn) / len(y_true)),
            'Balanced accuracy': (sens + spec) / 2}

def best_operating_point(y_true, probabilities, target_sensitivity):
    thresholds = np.unique(np.r_[0.0, probabilities, 1.0])
    rows = [classification_counts(y_true, probabilities, t) for t in thresholds]
    eligible = [r for r in rows if r['Sensitivity'] >= target_sensitivity]
    if not eligible:
        raise ValueError('No threshold meets target sensitivity.')
    return max(eligible, key=lambda r: (r['Specificity'], r['Accuracy'],
                                        r['Threshold'])), rows

class OperatingPointCheckpoint(keras.callbacks.Callback):
    def __init__(self, dataset, labels, path):
        super().__init__()
        self.dataset, self.labels, self.path = dataset, labels, path
        self.best_key = (-np.inf, -np.inf, -np.inf)
        self.rows = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        probabilities = self.model.predict(self.dataset, verbose=0).ravel()
        operating, _ = best_operating_point(
            self.labels, probabilities, TARGET_VALIDATION_SENSITIVITY)
        auc = float(roc_auc_score(self.labels, probabilities))
        logs['val_specificity_at_target'] = operating['Specificity']
        key = (operating['Specificity'], auc, operating['Accuracy'])
        self.rows.append({'epoch': epoch+1, 'AUC': auc, **operating})
        if key > self.best_key:
            self.best_key = key
            self.model.save(str(self.path))
        print(f" val specificity@target={operating['Specificity']:.4f}; "
              f"sensitivity={operating['Sensitivity']:.4f}; AUC={auc:.4f}")

def fit_classifier():
    keras.utils.set_random_seed(SEED)
    train_ds = classifier_dataset(train_df, 'dual', training=True)
    validation_ds = classifier_dataset(val_df, 'dual')
    labels = val_df.label.astype(int).to_numpy()
    model, _ = build_classifier('dual')
    head_path = MODELS_DIR / 'dual_balanced_brightness_head.keras'
    fine_path = MODELS_DIR / 'dual_balanced_brightness_fine.keras'
    checkpoint = OperatingPointCheckpoint(validation_ds, labels, head_path)
    compile_classifier(model, CLS_HEAD_LR)
    head_fit = model.fit(train_ds, validation_data=validation_ds, epochs=CLS_HEAD_EPOCHS,
              class_weight=CLASS_WEIGHTS,
              callbacks=[checkpoint, keras.callbacks.CSVLogger(str(RESULTS_DIR / "classifier_head_training_history.csv")),
                         keras.callbacks.EarlyStopping(
                  monitor='val_specificity_at_target', mode='max', patience=5)],
              verbose=1)
    pd.DataFrame(checkpoint.rows).to_csv(RESULTS_DIR / 'head_epoch_metrics.csv', index=False)
    (RESULTS_DIR / "classifier_head_training_history.json").write_text(
        json.dumps({k: [float(v) for v in values] for k, values in head_fit.history.items()}, indent=2))
    del model
    gc.collect()
    keras.backend.clear_session()

    model = keras.models.load_model(str(head_path))
    backbone = model.get_layer('efficientnetb0')
    backbone.trainable = True
    for layer in backbone.layers[:-20]:
        layer.trainable = False
    for layer in backbone.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    compile_classifier(model, CLS_FINE_TUNE_LR)
    checkpoint = OperatingPointCheckpoint(validation_ds, labels, fine_path)
    fine_fit = model.fit(train_ds, validation_data=validation_ds, epochs=CLS_FINE_TUNE_EPOCHS,
              class_weight=CLASS_WEIGHTS,
              callbacks=[checkpoint, keras.callbacks.CSVLogger(str(RESULTS_DIR / "classifier_fine_training_history.csv")),
                         keras.callbacks.EarlyStopping(
                  monitor='val_specificity_at_target', mode='max', patience=4)],
              verbose=1)
    pd.DataFrame(checkpoint.rows).to_csv(RESULTS_DIR / 'fine_epoch_metrics.csv', index=False)
    (RESULTS_DIR / "classifier_fine_training_history.json").write_text(
        json.dumps({k: [float(v) for v in values] for k, values in fine_fit.history.items()}, indent=2))
    del model
    gc.collect()
    keras.backend.clear_session()
    return [{'Stage': stage, 'Model path': str(path)} for stage, path in
            [('frozen head', head_path), ('fine-tuned', fine_path)]]

if MODE == 'train_new':
    print('Training with class weights:', CLASS_WEIGHTS)
    candidate_models = fit_classifier()
else:
    print('Using the frozen classifier. No classifier training in evaluate_saved mode.')


## 14. Locked classifier configuration
The frozen candidate uses its saved threshold. New experiments select on validation only.

In [ ]:
def predict_with_flips(model, frame, four_view=False):
    # classifier_dataset does not shuffle evaluation rows; the predictions
    # retain the same order as frame. Augmentation layers are inactive at predict.
    base_ds = classifier_dataset(frame, 'dual', training=False)
    if not four_view:
        return model.predict(base_ds, verbose=0).ravel()
    predictions = []
    for flip_lr, flip_ud in [(False, False), (True, False),
                             (False, True), (True, True)]:
        def apply_flip(features, target, flip_lr=flip_lr, flip_ud=flip_ud):
            views = {}
            for key, pixels in features.items():
                if flip_lr:
                    pixels = tf.image.flip_left_right(pixels)
                if flip_ud:
                    pixels = tf.image.flip_up_down(pixels)
                views[key] = pixels
            return views, target
        flip_ds = base_ds.map(apply_flip, num_parallel_calls=AUTOTUNE)
        predictions.append(model.predict(flip_ds, verbose=0).ravel())
    return np.mean(np.stack(predictions), axis=0)

if MODE == "train_new":
    y_val = val_df['label'].astype(int).to_numpy()
    selection_rows = []
    all_threshold_rows = []
    candidate_probabilities = {}
    for candidate in candidate_models:
        model = keras.models.load_model(candidate['Model path'])
        for method, four_view in [('single', False), ('4-flip average', True)]:
            probabilities = predict_with_flips(model, val_df, four_view=four_view)
            key = (candidate['Stage'], method)
            candidate_probabilities[key] = probabilities
            auc = float(roc_auc_score(y_val, probabilities))
            best, rows = best_operating_point(
                y_val, probabilities, TARGET_VALIDATION_SENSITIVITY)
            for row in rows:
                all_threshold_rows.append({'Stage': candidate['Stage'],
                                           'Method': method,
                                           'Validation AUC': auc, **row})
            selection_rows.append({'Stage': candidate['Stage'], 'Method': method,
                                   'Model path': candidate['Model path'],
                                   'Validation AUC': auc, **best})
        del model
        gc.collect()

    selection_table = pd.DataFrame(selection_rows).sort_values(
        ['Specificity', 'Validation AUC', 'Accuracy'], ascending=False
    ).reset_index(drop=True)
    display(selection_table.drop(columns='Model path').round(4))
    winner = selection_table.iloc[0]
    SELECTED_STAGE = str(winner['Stage'])
    SELECTED_METHOD = str(winner['Method'])
    SELECTED_THRESHOLD = float(winner['Threshold'])
    SELECTED_VIEW = 'dual'
    selected_classifier = keras.models.load_model(winner['Model path'])
    print('Selected on validation:', SELECTED_STAGE, SELECTED_METHOD,
          'threshold', SELECTED_THRESHOLD)

    selection_table.to_csv(RESULTS_DIR / 'validation_candidate_comparison.csv', index=False)
    pd.DataFrame(all_threshold_rows).to_csv(
        RESULTS_DIR / 'validation_all_thresholds.csv', index=False)
    selected_val_prob = candidate_probabilities[(SELECTED_STAGE, SELECTED_METHOD)]
    val_predictions = val_df[['image_id', 'diagnosis', 'label', 'image_path',
                              'roi_path', 'pred_mask_path']].copy()
    val_predictions['malignancy_probability'] = selected_val_prob
    val_predictions['predicted_label'] = (
        selected_val_prob >= SELECTED_THRESHOLD).astype(int)
    val_predictions['outcome'] = [
        ('TP' if truth == 1 and pred == 1 else
         'TN' if truth == 0 and pred == 0 else
         'FP' if truth == 0 and pred == 1 else 'FN')
        for truth, pred in zip(y_val, val_predictions['predicted_label'])
    ]
    val_predictions.to_csv(RESULTS_DIR / 'selected_validation_predictions.csv',
                           index=False)
    selected_model_path = MODELS_DIR / 'selected_final_classifier.keras'
    selected_classifier.save(str(selected_model_path))
    (RESULTS_DIR / 'selected_configuration.json').write_text(json.dumps({
        'view': SELECTED_VIEW, 'class_weights': CLASS_WEIGHTS,
        'stage': SELECTED_STAGE, 'method': SELECTED_METHOD,
        'threshold': SELECTED_THRESHOLD,
        'target_validation_sensitivity': TARGET_VALIDATION_SENSITIVITY,
        'segmenter_reused': SEGMENTER_REUSED,
        'segmentation_method': SELECTED_SEGMENTATION_METHOD,
        'segmentation_threshold': SEG_THRESHOLD,
        'roi_padding': 0.15, 'split_sha256': split_hash,
        'note': 'Official test previously viewed; final follow-up is exploratory.',
    }, indent=2))
else:
    SELECTED_STAGE = SAVED_CONFIGURATION['stage']
    SELECTED_METHOD = SAVED_CONFIGURATION['method']
    SELECTED_THRESHOLD = float(SAVED_CONFIGURATION['threshold'])
    SELECTED_VIEW = SAVED_CONFIGURATION['view']
    selected_model_path = MODELS_DIR / 'selected_final_classifier.keras'
    selected_classifier = keras.models.load_model(selected_model_path, compile=False)
    selected_val_prob = predict_with_flips(selected_classifier, val_df,
        four_view=(SELECTED_METHOD == '4-flip average'))
    comparison = RESULTS_DIR / 'validation_candidate_comparison.csv'
    selection_table = pd.read_csv(comparison) if comparison.is_file() else pd.DataFrame()
    if len(selection_table):
        display(selection_table.drop(columns='Model path', errors='ignore').round(4))
    (RESULTS_DIR / 'selected_configuration.json').write_text(json.dumps(SAVED_CONFIGURATION, indent=2))
    print('Frozen classifier:', SELECTED_STAGE, SELECTED_METHOD, SELECTED_THRESHOLD)


## 15. Official-test classification
Exploratory follow-up on the same 379 images. No model selection occurs here.

In [ ]:
official_test_ds = classifier_dataset(test_df, SELECTED_VIEW)
y_test = test_df['label'].astype(int).to_numpy()
test_prob = predict_with_flips(
    selected_classifier, test_df, four_view=(SELECTED_METHOD == '4-flip average'))
test_metrics = classification_counts(y_test, test_prob, SELECTED_THRESHOLD)
test_metrics['ROC-AUC'] = float(roc_auc_score(y_test, test_prob))
test_metrics['Precision'] = float(precision_score(y_test, (test_prob >= SELECTED_THRESHOLD).astype(int), zero_division=0))
test_metrics['F1'] = float(f1_score(y_test,
                                  (test_prob >= SELECTED_THRESHOLD).astype(int)))
test_metrics['Input'] = SELECTED_VIEW
test_metrics['Method'] = SELECTED_METHOD
test_metrics['Class weights'] = 'balanced'
test_metrics['Stage'] = SELECTED_STAGE
test_metrics['N'] = len(test_df)
official_test_metrics = pd.DataFrame([test_metrics])
display(official_test_metrics.T.rename(columns={0: 'Exploratory follow-up result'}))
official_test_metrics.to_csv(RESULTS_DIR / 'FOLLOWUP_official_test_classification_metrics.csv',
                             index=False)
test_predictions = test_df[['image_id', 'diagnosis', 'label', 'image_path',
                            'roi_path', 'pred_mask_path']].copy()
test_predictions['malignancy_probability'] = test_prob
test_predictions['predicted_label'] = (test_prob >= SELECTED_THRESHOLD).astype(int)
test_predictions['outcome'] = [
    ('TP' if truth == 1 and pred == 1 else
     'TN' if truth == 0 and pred == 0 else
     'FP' if truth == 0 and pred == 1 else 'FN')
    for truth, pred in zip(y_test, test_predictions['predicted_label'])]
test_predictions.to_csv(RESULTS_DIR / 'FOLLOWUP_official_test_predictions.csv', index=False)

matrix = confusion_matrix(y_test, test_predictions['predicted_label'], labels=[0, 1])
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].imshow(matrix, cmap='Blues')
axes[0].set_xticks([0, 1], labels=['Benign', 'Melanoma'])
axes[0].set_yticks([0, 1], labels=['Benign', 'Melanoma'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Exploratory follow-up confusion matrix')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(matrix[i, j]), ha='center', va='center', fontsize=16)
fpr, tpr, _ = roc_curve(y_test, test_prob)
axes[1].plot(fpr, tpr, label=f"AUC = {test_metrics['ROC-AUC']:.3f}")
axes[1].plot([0, 1], [0, 1], '--', color='grey')
axes[1].set_xlabel('False positive rate'); axes[1].set_ylabel('Sensitivity')
axes[1].set_title('Exploratory follow-up ROC'); axes[1].legend()
plt.tight_layout(); plt.show()


## 16. Complete test loss, metrics and comparisons
Pixel sensitivity/specificity assess lesion/background segmentation; classification sensitivity/specificity assess melanoma/benign labels. The app must reproduce the notebook scores and masks.

In [ ]:
configuration = json.loads((RESULTS_DIR / 'selected_configuration.json').read_text())
pipeline = MelanomaPipeline(selected_segmenter, selected_classifier, configuration, OUTPUT_DIR)
segmentation_metrics, classification_metrics_complete, segmentation_test = evaluate_pipeline(
    pipeline, test_df, test_prob, OUTPUT_DIR, validation_segmentation=segmentation_selection)
display(pd.DataFrame([segmentation_metrics]).T.rename(columns={0: 'Segmentation result'}))
display(pd.DataFrame([classification_metrics_complete]).T.rename(columns={0: 'Classification result'}))


## 17. Training curves and HTML report
Historical figures show recorded console summaries. Exact histories are used whenever saved CSVs are available. Classifier curve accuracy is measured at 0.5; the final confusion matrix uses the selected threshold.

In [ ]:
history_notes = []
histories = [
    ('segmentation', 'segmentation_training_history.csv', 'segmentation_training_curves',
     'Attention U-Net segmentation training', True),
    ('classifier_head', 'classifier_head_training_history.csv', 'classifier_head_training_curves',
     'EfficientNetB0: frozen head training', False),
    ('classifier_fine', 'classifier_fine_training_history.csv', 'classifier_fine_training_curves',
     'EfficientNetB0: fine-tuning experiment', False),
]
for key, filename, figure_name, title, is_segmentation in histories:
    exact_path = RESULTS_DIR / filename
    if exact_path.is_file():
        plot_training_history(pd.read_csv(exact_path), OUTPUT_DIR, figure_name, title,
                              segmentation=is_segmentation)
        history_notes.append(f'{key}: exact saved epoch history.')
    elif MODE == 'evaluate_saved' and str(SAVED_RUN_DIR) == RECOVERED_HISTORY['source_run']:
        historical = pd.DataFrame(RECOVERED_HISTORY[key])
        historical['history_source'] = 'rounded historical console summary'
        historical.to_csv(RESULTS_DIR / ('historical_' + filename), index=False)
        provenance = 'Historical rounded console summaries; not a new training run'
        plot_training_history(historical, OUTPUT_DIR, 'historical_' + figure_name,
                              title + ' (historical)', segmentation=is_segmentation,
                              provenance=provenance)
        history_notes.append(f'{key}: rounded historical console summaries.')
    else:
        history_notes.append(f'{key}: history unavailable; no curve fabricated.')
history_notes.append(RECOVERED_HISTORY['note'] if MODE == 'evaluate_saved' and
                     str(SAVED_RUN_DIR) == RECOVERED_HISTORY['source_run'] else
                     'Exact epoch CSVs are saved automatically when MODE is train_new.')
history_notes = ' '.join(history_notes)
(RESULTS_DIR / 'training_history_provenance.json').write_text(json.dumps({
    'mode': MODE, 'source_run': str(SAVED_RUN_DIR) if MODE == 'evaluate_saved' else str(OUTPUT_DIR),
    'note': history_notes,
}, indent=2))
print(history_notes)


## 18. Load the evaluation outputs from this run

The confidence-interval analysis reads `OUTPUT_DIR`, created in Section 3. It does not use a hard-coded historical reporting folder or remount Drive. Sections 15–17 must finish first. The bootstrap settings and optional comparison folders are also configured in Section 3.

In [ ]:
from IPython.display import display

RUN_DIR = Path(OUTPUT_DIR)
TABLES_DIR = RUN_DIR / 'tables'
FIGURES_DIR = RUN_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CLASSIFICATION_PREDICTIONS = TABLES_DIR / 'FOLLOWUP_official_test_predictions.csv'
SEGMENTATION_PER_IMAGE = TABLES_DIR / 'complete_segmentation_per_image.csv'
CLASSIFICATION_METRICS = TABLES_DIR / 'complete_classification_test_metrics.csv'
SEGMENTATION_METRICS = TABLES_DIR / 'complete_segmentation_test_metrics.csv'

required = [CLASSIFICATION_PREDICTIONS, SEGMENTATION_PER_IMAGE,
            CLASSIFICATION_METRICS, SEGMENTATION_METRICS]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('Run Sections 15–17 first. Missing outputs:\n' + '\n'.join(missing))

def validate_run_frames(cls_frame, seg_frame, threshold_value, run_name):
    required_cls = {'image_id', 'label', 'malignancy_probability', 'predicted_label'}
    required_seg = {'image_id', 'dice', 'iou', 'test_loss',
                    'pixel_TP', 'pixel_TN', 'pixel_FP', 'pixel_FN'}
    for frame, columns, task in [(cls_frame, required_cls, 'classification'),
                                  (seg_frame, required_seg, 'segmentation')]:
        if not columns.issubset(frame.columns):
            raise ValueError(f'{run_name}: missing {task} columns {sorted(columns - set(frame.columns))}')
        if len(frame) < 2 or frame.image_id.isna().any() or not frame.image_id.is_unique:
            raise ValueError(f'{run_name}: {task} needs at least two rows with unique, nonmissing image IDs.')
    if set(cls_frame.image_id) != set(seg_frame.image_id):
        raise ValueError(f'{run_name}: classification and segmentation image IDs differ.')
    if not np.isfinite(threshold_value) or not 0 <= threshold_value <= 1:
        raise ValueError(f'{run_name}: invalid saved classifier threshold.')
    for col in ['label', 'predicted_label', 'malignancy_probability']:
        cls_frame[col] = pd.to_numeric(cls_frame[col], errors='raise')
    if set(cls_frame.label.unique()) != {0, 1}:
        raise ValueError(f'{run_name}: classification labels must contain both benign (0) and melanoma (1).')
    scores = cls_frame.malignancy_probability.to_numpy(dtype=float)
    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError(f'{run_name}: probabilities must be finite values in [0, 1].')
    if not np.array_equal(cls_frame.predicted_label.to_numpy(), (scores >= threshold_value).astype(int)):
        raise ValueError(f'{run_name}: saved predicted labels do not match the recorded threshold.')
    for col in sorted(required_seg - {'image_id'}):
        seg_frame[col] = pd.to_numeric(seg_frame[col], errors='raise')
    numbers = seg_frame[sorted(required_seg - {'image_id'})].to_numpy(dtype=float)
    if not np.isfinite(numbers).all() or (numbers < 0).any():
        raise ValueError(f'{run_name}: segmentation metrics/counts must be finite and nonnegative.')
    if (seg_frame[['dice', 'iou']].to_numpy() > 1).any():
        raise ValueError(f'{run_name}: Dice and IoU must be in [0, 1].')
    counts = seg_frame[['pixel_TP', 'pixel_TN', 'pixel_FP', 'pixel_FN']].to_numpy(dtype=float)
    if not np.equal(counts, np.floor(counts)).all():
        raise ValueError(f'{run_name}: pixel confusion counts must be integers.')
    if counts[:, [0, 3]].sum() <= 0 or counts[:, [1, 2]].sum() <= 0:
        raise ValueError(f'{run_name}: pooled pixel sensitivity/specificity have empty denominators.')

cls = pd.read_csv(CLASSIFICATION_PREDICTIONS)
seg = pd.read_csv(SEGMENTATION_PER_IMAGE)
cls_summary = pd.read_csv(CLASSIFICATION_METRICS).iloc[0]
seg_summary = pd.read_csv(SEGMENTATION_METRICS).iloc[0]
threshold = float(cls_summary['Threshold'])
validate_run_frames(cls, seg, threshold, 'Current run')
if not isinstance(BOOTSTRAP_REPLICATES, int) or BOOTSTRAP_REPLICATES < 100:
    raise ValueError('BOOTSTRAP_REPLICATES must be an integer of at least 100; use 5000 for reporting.')
print(f'Loaded {len(cls)} paired evaluation cases from {RUN_DIR}; classifier threshold={threshold:.6f}.')
print(f'Melanoma={int(cls.label.sum())}; benign={int((cls.label == 0).sum())}.')
print('The evaluation models and saved thresholds are unchanged by this analysis.')


## 19. Classification results and confidence intervals

Accuracy, sensitivity, specificity and precision receive Wilson intervals. F1-score, balanced accuracy, ROC-AUC and unweighted binary cross-entropy receive stratified case-level bootstrap percentile intervals. Resampling is stratified by reference diagnosis so both classes remain represented in every replicate.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
y = cls['label'].to_numpy(dtype=int)
prob = cls['malignancy_probability'].to_numpy(dtype=float)
pred = cls['predicted_label'].to_numpy(dtype=int)

def wilson_interval(successes, n, z=1.959963984540054):
    if n <= 0:
        return (np.nan, np.nan)
    p = successes / n
    den = 1 + z*z/n
    centre = (p + z*z/(2*n)) / den
    half = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / den
    return centre-half, centre+half

def classification_stats(y_true, p_score, threshold):
    y_true = np.asarray(y_true, dtype=int)
    p_score = np.asarray(p_score, dtype=float)
    pred_label = (p_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred_label, labels=[0,1]).ravel()
    sens = tp/(tp+fn) if tp+fn else np.nan
    spec = tn/(tn+fp) if tn+fp else np.nan
    prec = tp/(tp+fp) if tp+fp else 0.0
    f1 = 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0
    clipped = np.clip(p_score, 1e-7, 1-1e-7)
    loss = np.mean(-y_true*np.log(clipped) - (1-y_true)*np.log(1-clipped))
    return {
        'Accuracy': (tp+tn)/len(y_true), 'Sensitivity': sens,
        'Specificity': spec, 'Precision': prec, 'F1-score': f1,
        'Balanced accuracy': (sens+spec)/2,
        'ROC-AUC': roc_auc_score(y_true, p_score),
        'Classification test loss (unweighted BCE)': loss,
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    }

point = classification_stats(y, prob, threshold)
idx0 = np.flatnonzero(y == 0)
idx1 = np.flatnonzero(y == 1)
boot_keys = ['Accuracy','Sensitivity','Specificity','Precision','F1-score',
             'Balanced accuracy','ROC-AUC','Classification test loss (unweighted BCE)']
boot = {key: [] for key in boot_keys}
for _ in range(BOOTSTRAP_REPLICATES):
    sample = np.concatenate([
        rng.choice(idx0, size=len(idx0), replace=True),
        rng.choice(idx1, size=len(idx1), replace=True)
    ])
    values = classification_stats(y[sample], prob[sample], threshold)
    for key in boot_keys:
        boot[key].append(values[key])

proportion_denominators = {
    'Accuracy': (point['TP']+point['TN'], len(y)),
    'Sensitivity': (point['TP'], point['TP']+point['FN']),
    'Specificity': (point['TN'], point['TN']+point['FP']),
    'Precision': (point['TP'], point['TP']+point['FP'])
}
rows=[]
for key in boot_keys:
    vals=np.asarray(boot[key],dtype=float)
    lo,hi=np.quantile(vals,[0.025,0.975])
    method='Stratified image-level percentile bootstrap'
    if key in proportion_denominators:
        lo,hi=wilson_interval(*proportion_denominators[key])
        method='Wilson score interval'
    rows.append({'Metric':key,'Estimate':point[key],'95% CI lower':lo,
                 '95% CI upper':hi,'Interval method':method})
classification_ci = pd.DataFrame(rows)
display(classification_ci.round(4))
classification_ci.to_csv(TABLES_DIR/'supplemental_classification_confidence_intervals.csv',index=False)
print('Confusion counts:', {k:point[k] for k in ['TP','TN','FP','FN']})


## 20. Segmentation results and image-level intervals

Bootstrap resampling is performed over images (not pixels), preserving the image as the sampling unit. Pixel sensitivity and specificity are recomputed from the per-image pixel counts within each bootstrap sample. This avoids treating millions of correlated pixels as independent observations.

In [ ]:
seg = seg.copy()
for col in ['dice','iou','test_loss','pixel_TP','pixel_TN','pixel_FP','pixel_FN']:
    seg[col] = pd.to_numeric(seg[col], errors='raise')
seg = seg.reset_index(drop=True)
N = len(seg)
if N < 2:
    raise ValueError('At least two per-image rows are required for bootstrap intervals.')

seg_values = {key: [] for key in ['Mean Dice','Median Dice','Mean IoU','Mean segmentation test loss',
                                   'Pooled pixel sensitivity','Pooled pixel specificity']}
for _ in range(BOOTSTRAP_REPLICATES):
    b = seg.iloc[rng.integers(0,N,size=N)]
    tp_px,tn_px,fp_px,fn_px = [b[c].sum() for c in ['pixel_TP','pixel_TN','pixel_FP','pixel_FN']]
    seg_values['Mean Dice'].append(b.dice.mean())
    seg_values['Median Dice'].append(b.dice.median())
    seg_values['Mean IoU'].append(b.iou.mean())
    seg_values['Mean segmentation test loss'].append(b.test_loss.mean())
    seg_values['Pooled pixel sensitivity'].append(tp_px/(tp_px+fn_px) if tp_px+fn_px else np.nan)
    seg_values['Pooled pixel specificity'].append(tn_px/(tn_px+fp_px) if tn_px+fp_px else np.nan)

point_seg = {
    'Mean Dice':seg.dice.mean(), 'Median Dice':seg.dice.median(),
    'Mean IoU':seg.iou.mean(), 'Mean segmentation test loss':seg.test_loss.mean(),
    'Pooled pixel sensitivity':seg.pixel_TP.sum()/(seg.pixel_TP.sum()+seg.pixel_FN.sum()),
    'Pooled pixel specificity':seg.pixel_TN.sum()/(seg.pixel_TN.sum()+seg.pixel_FP.sum())
}
seg_rows=[]
for key,val in point_seg.items():
    lo,hi=np.nanquantile(seg_values[key],[0.025,0.975])
    seg_rows.append({'Metric':key,'Estimate':val,'95% CI lower':lo,'95% CI upper':hi,
                     'Interval method':'Image-level percentile bootstrap'})
segmentation_ci=pd.DataFrame(seg_rows)
display(segmentation_ci.round(4))
segmentation_ci.to_csv(TABLES_DIR/'supplemental_segmentation_confidence_intervals.csv',index=False)
print(f'Images: {N}; zero-Dice cases: {(seg.dice == 0).sum()}; median Dice: {seg.dice.median():.4f}.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4.5))
axes[0].hist(seg.dice, bins=np.linspace(0,1,21), color='#2b7894', edgecolor='white')
axes[0].axvline(seg.dice.mean(), color='#c43c35', linestyle='--', label=f'Mean {seg.dice.mean():.3f}')
axes[0].axvline(seg.dice.median(), color='#2b5d3c', linestyle=':', label=f'Median {seg.dice.median():.3f}')
axes[0].set(xlabel='Per-image Dice', ylabel='Number of images', title='Segmentation Dice distribution')
axes[0].legend()
cm=confusion_matrix(y,pred,labels=[0,1])
axes[1].imshow(cm,cmap='Blues')
axes[1].set_xticks([0,1],labels=['Benign','Melanoma'])
axes[1].set_yticks([0,1],labels=['Benign','Melanoma'])
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Reference')
axes[1].set_title(f'Classification at cutoff {threshold:.4f}')
for i in range(2):
    for j in range(2): axes[1].text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=14)
fig.tight_layout()
fig.savefig(FIGURES_DIR/'supplemental_dice_and_confusion.png',dpi=220,bbox_inches='tight')
plt.show()

## 21. Optional paired comparisons with earlier saved runs

Configure `PRIOR_RUN_DIRS` in Section 3. Each earlier folder needs the same standard per-image CSV files under `tables/` and its own `complete_classification_test_metrics.csv` with the saved threshold. No threshold is substituted from the current run.

The summary reports all configured runs descriptively. Paired confidence intervals are calculated only for runs with identical classification and segmentation image IDs and matching diagnosis labels. The same sampled images are used in both members of a pair. Classification differences use class-stratified image resampling; segmentation differences use image resampling. Differences are **variant minus current baseline**. Negative loss differences favour the variant; positive accuracy, AUC or overlap differences favour the variant.

Matching images supports pairing but does not prove that preprocessing, reference masks, training data or selection procedures were controlled. Hair-removal or architecture comparisons need documented experimental settings. Aggregate-only historical results cannot provide paired intervals. All comparisons are exploratory; the displayed intervals have no multiplicity adjustment.

In [ ]:
def load_run(run_dir):
    t=Path(run_dir)/'tables'
    pred_file=t/'FOLLOWUP_official_test_predictions.csv'
    seg_file=t/'complete_segmentation_per_image.csv'
    if not pred_file.is_file() or not seg_file.is_file():
        raise FileNotFoundError(f'{run_dir} must contain {pred_file.name} and {seg_file.name} in tables/.')
    return pd.read_csv(pred_file),pd.read_csv(seg_file)

def summarize_run(name, cls_df, seg_df, threshold_value):
    m=classification_stats(cls_df.label.to_numpy(dtype=int),
                          cls_df.malignancy_probability.to_numpy(dtype=float),
                          float(threshold_value))
    return {'Run':name,'N classification':len(cls_df),'N segmentation':len(seg_df),
            'Mean Dice':seg_df.dice.mean(),'Mean IoU':seg_df.iou.mean(),
            'Accuracy':m['Accuracy'],'Sensitivity':m['Sensitivity'],
            'Specificity':m['Specificity'],'ROC-AUC':m['ROC-AUC'],
            'Threshold':float(threshold_value)}

def paired_bootstrap_deltas(base, variant, value_fn_base, value_fn_variant, keys, n_boot=BOOTSTRAP_REPLICATES):
    """Case-paired, class-stratified bootstrap; returns variant minus baseline."""
    base=base.reset_index(drop=True)
    variant=variant.reset_index(drop=True)
    labels=base.label.to_numpy(dtype=int)
    i0=np.flatnonzero(labels==0); i1=np.flatnonzero(labels==1)
    point_base=value_fn_base(base); point_variant=value_fn_variant(variant)
    samples={k:[] for k in keys}
    for _ in range(n_boot):
        ix=np.concatenate([rng.choice(i0,size=len(i0),replace=True),
                           rng.choice(i1,size=len(i1),replace=True)])
        vb=value_fn_base(base.iloc[ix]); vv=value_fn_variant(variant.iloc[ix])
        for k in keys: samples[k].append(vv[k]-vb[k])
    rows=[]
    for k in keys:
        d=point_variant[k]-point_base[k]
        lo,hi=np.quantile(samples[k],[.025,.975])
        rows.append({'Metric':k,'Variant minus baseline':d,'95% paired bootstrap CI lower':lo,
                     '95% paired bootstrap CI upper':hi})
    return rows

baseline_threshold=threshold
baseline_name = 'Current frozen Attention U-Net' if MODE == 'evaluate_saved' else 'Current newly trained Attention U-Net'
comparison_rows=[summarize_run(baseline_name,cls,seg,baseline_threshold)]
paired_metric_rows=[]
paired_summary=pd.DataFrame()
for name,run_dir in PRIOR_RUN_DIRS.items():
    if run_dir is None:
        continue
    other_cls,other_seg=load_run(run_dir)
    metrics_path=Path(run_dir)/'tables'/'complete_classification_test_metrics.csv'
    if not metrics_path.is_file():
        raise FileNotFoundError(f'{name}: missing saved classifier metrics/threshold: {metrics_path}')
    other_threshold=float(pd.read_csv(metrics_path).iloc[0]['Threshold'])
    validate_run_frames(other_cls, other_seg, other_threshold, name)
    comparison_rows.append(summarize_run(name,other_cls,other_seg,other_threshold))

    base_cls=cls.sort_values('image_id').reset_index(drop=True)
    var_cls=other_cls.sort_values('image_id').reset_index(drop=True)
    same_cls=(base_cls.image_id.tolist()==var_cls.image_id.tolist() and
              np.array_equal(base_cls.label.to_numpy(dtype=int),var_cls.label.to_numpy(dtype=int)))
    base_seg=seg.sort_values('image_id').reset_index(drop=True)
    var_seg=other_seg.sort_values('image_id').reset_index(drop=True)
    same_seg=same_cls and base_seg.image_id.tolist()==var_seg.image_id.tolist()

    if same_cls:
        def base_cls_fn(frame):
            return classification_stats(frame.label.to_numpy(dtype=int),
                frame.malignancy_probability.to_numpy(dtype=float),baseline_threshold)
        def variant_cls_fn(frame):
            return classification_stats(frame.label.to_numpy(dtype=int),
                frame.malignancy_probability.to_numpy(dtype=float),other_threshold)
        paired_metric_rows.extend([{'Comparison':name+' vs current',**row} for row in paired_bootstrap_deltas(
            base_cls,var_cls,base_cls_fn,variant_cls_fn,['Accuracy','Sensitivity','Specificity','Precision','F1-score',
            'Balanced accuracy','ROC-AUC','Classification test loss (unweighted BCE)'])])
    else:
        print(f'{name}: classification IDs or labels differ; descriptive summary only.')

    if same_seg:
        def seg_fn(frame):
            tp_px=frame.pixel_TP.sum(); tn_px=frame.pixel_TN.sum()
            fp_px=frame.pixel_FP.sum(); fn_px=frame.pixel_FN.sum()
            return {'Mean Dice':frame.dice.mean(),'Mean IoU':frame.iou.mean(),
                    'Mean segmentation test loss':frame.test_loss.mean(),
                    'Pooled pixel sensitivity':tp_px/(tp_px+fn_px),
                    'Pooled pixel specificity':tn_px/(tn_px+fp_px)}
        # The paired image IDs are aligned before resampling. The same bootstrap sample
        # is applied to both runs, preserving the paired comparison.
        b=base_seg.reset_index(drop=True); v=var_seg.reset_index(drop=True)
        boot_seg={k:[] for k in seg_fn(b)}
        for _ in range(BOOTSTRAP_REPLICATES):
            ix=rng.integers(0,len(b),size=len(b))
            sb=seg_fn(b.iloc[ix]); sv=seg_fn(v.iloc[ix])
            for k in boot_seg: boot_seg[k].append(sv[k]-sb[k])
        sb=seg_fn(b); sv=seg_fn(v)
        for k in boot_seg:
            lo,hi=np.quantile(boot_seg[k],[.025,.975])
            paired_metric_rows.append({'Comparison':name+' vs current','Metric':k,
                'Variant minus baseline':sv[k]-sb[k],
                '95% paired bootstrap CI lower':lo,'95% paired bootstrap CI upper':hi})
    else:
        print(f'{name}: image IDs or diagnosis labels do not match; no paired segmentation CIs.')

comparison_summary=pd.DataFrame(comparison_rows)
display(comparison_summary)
comparison_summary.to_csv(TABLES_DIR/'supplemental_run_comparison_summary.csv',index=False)
if paired_metric_rows:
    paired_summary=pd.DataFrame(paired_metric_rows)
    display(paired_summary)
    paired_summary.to_csv(TABLES_DIR/'supplemental_paired_comparison_deltas.csv',index=False)
else:
    stale_paired_output=TABLES_DIR/'supplemental_paired_comparison_deltas.csv'
    if stale_paired_output.is_file():
        stale_paired_output.unlink()
    print('No paired comparison outputs produced. Configure prior run folders and confirm saved per-image CSVs.')


## 22. Training-history and software-evidence provenance

This check reports what the saved run recorded. It does not reconstruct or certify training curves. If exact per-epoch CSVs are absent and the provenance file says values came from rounded historical console summaries, label those plots as historical and do not use them to explain the current checkpoint's performance.

In [ ]:
provenance_path=TABLES_DIR/'training_history_provenance.json'
if provenance_path.is_file():
    provenance=json.loads(provenance_path.read_text())
    print('Training-history mode:',provenance.get('mode'))
    print('Source run:',provenance.get('source_run'))
    print('Provenance note:',provenance.get('note'))
else:
    print('No training-history provenance file found.')
for filename in ['segmentation_training_history.csv','classifier_head_training_history.csv','classifier_fine_training_history.csv']:
    f=TABLES_DIR/filename
    print(f'{filename}:', 'exact CSV available' if f.is_file() else 'not available in this run folder')
parity=TABLES_DIR/'app_notebook_parity.csv'
if parity.is_file():
    parity_df=pd.read_csv(parity)
    print(f'Notebook–app parity cases recorded: {len(parity_df)}')
    display(parity_df)
else:
    print('No app_notebook_parity.csv found; do not claim parity testing from this run.')

## 23. Results paragraph from the calculated outputs

This cell writes a paragraph from the current run's observed metrics and intervals. Check the dataset description and experimental context before inserting it in your report. The intervals describe this dataset, conditional on the saved model and threshold; they are not independent clinical validation.

In [ ]:
class_ci=classification_ci.set_index('Metric')
seg_ci=segmentation_ci.set_index('Metric')
def ci_text(frame,metric,digits=3):
    row=frame.loc[metric]
    return f"{row['Estimate']:.{digits}f} (95% CI {row['95% CI lower']:.{digits}f}–{row['95% CI upper']:.{digits}f})"
def pct_text(frame,metric):
    row=frame.loc[metric]
    return f"{100*row['Estimate']:.1f}% (95% CI {100*row['95% CI lower']:.1f}–{100*row['95% CI upper']:.1f}%)"

n_cases=len(cls); n_melanoma=int(y.sum()); n_benign=int((y==0).sum())
tp,tn,fp,fn=[point[k] for k in ['TP','TN','FP','FN']]
model_description = 'frozen Attention U-Net' if MODE == 'evaluate_saved' else 'newly trained Attention U-Net'
results_paragraph=(
 f"On the {n_cases}-image ISIC 2016 Task 3B evaluation set ({n_melanoma} melanoma and {n_benign} benign images), "
 f"the {model_description} achieved a mean per-image Dice of {ci_text(seg_ci,'Mean Dice')} and mean IoU of {ci_text(seg_ci,'Mean IoU')}. "
 f"Median Dice was {seg.dice.median():.4f}, and {(seg.dice==0).sum()} image(s) had Dice 0. "
 f"At the validation-selected classifier cutoff of {threshold:.4f}, the model produced {int(tp)} true positives, {int(tn)} true negatives, {int(fp)} false positives and {int(fn)} false negatives. "
 f"Accuracy was {pct_text(class_ci,'Accuracy')}, sensitivity {pct_text(class_ci,'Sensitivity')}, specificity {pct_text(class_ci,'Specificity')}, "
 f"and precision {pct_text(class_ci,'Precision')}; ROC-AUC was {ci_text(class_ci,'ROC-AUC')}. "
 f"Classification test loss (unweighted binary cross-entropy) was {ci_text(class_ci,'Classification test loss (unweighted BCE)',4)}. "
 f"These are exploratory follow-up results because the official test set had been inspected during earlier development; they do not constitute independent clinical validation."
)
print(results_paragraph)
(Tables_path:=TABLES_DIR/'supplemental_results_text.txt').write_text(results_paragraph+'\n', encoding='utf-8')
print('Saved:',Tables_path)


## 24. Build the combined HTML report

The report includes the original evaluation figures, the supplemental figure, confidence-interval tables, descriptive run comparisons, any available paired comparisons and the generated Results paragraph. Bootstrap settings and hashes of the analysed CSVs are recorded alongside the outputs.

In [ ]:
import html as html_tools

analysis_provenance = {
    'run_dir': str(RUN_DIR), 'mode': MODE,
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'random_seed': RANDOM_SEED,
    'confidence_level': 0.95,
    'classification_threshold': threshold,
    'sampling_unit': 'image; no patient-cluster adjustment',
    'classification_bootstrap': 'class-stratified image-level percentile bootstrap',
    'classification_proportion_intervals': 'Wilson score',
    'segmentation_bootstrap': 'image-level percentile bootstrap',
    'paired_difference': 'variant minus current baseline',
    'multiplicity_adjusted': False,
    'prior_run_dirs': {name: str(path) if path is not None else None
                       for name, path in PRIOR_RUN_DIRS.items()},
    'input_sha256': {path.name: hashlib.sha256(path.read_bytes()).hexdigest()
                     for path in required},
    'interpretation': 'Exploratory follow-up on a previously inspected official test set.',
}
(TABLES_DIR/'supplemental_analysis_provenance.json').write_text(
    json.dumps(analysis_provenance, indent=2), encoding='utf-8')

report_path = make_html_report(OUTPUT_DIR, segmentation_metrics,
                              classification_metrics_complete, history_notes)
extra_sections = [
    '<section id="supplemental-analysis"><h2>Confidence intervals and earlier-run comparisons</h2>',
    '<p>95% Wilson intervals are used for classification proportions; other classification intervals '
    'use class-stratified image bootstrap resampling. Segmentation intervals resample images. '
    f'Bootstrap replicates: {BOOTSTRAP_REPLICATES}; random seed: {RANDOM_SEED}. '
    'These intervals do not remove test-set reuse or within-patient dependence.</p>',
    '<h3>Classification confidence intervals</h3>',
    classification_ci.to_html(index=False, float_format=lambda value: f'{value:.5f}'),
    '<h3>Segmentation confidence intervals</h3>',
    segmentation_ci.to_html(index=False, float_format=lambda value: f'{value:.5f}'),
    '<h3>Descriptive run comparison</h3>',
    comparison_summary.to_html(index=False, float_format=lambda value: f'{value:.5f}'),
]
if not paired_summary.empty:
    extra_sections += ['<h3>Paired differences: variant minus baseline</h3>',
                       '<p>Exploratory 95% intervals without multiplicity adjustment. '
                       'Check experimental provenance before attributing a difference to one pipeline change.</p>',
                       paired_summary.to_html(index=False, float_format=lambda value: f'{value:.5f}')]
else:
    extra_sections.append('<p>No paired comparison was configured or no comparable paired run was available.</p>')
extra_sections += ['<h3>Results paragraph</h3><p>' + html_tools.escape(results_paragraph) + '</p></section>']
report_text = report_path.read_text(encoding='utf-8')
if '</body>' not in report_text:
    raise ValueError('The generated HTML report has no closing body tag.')
report_path.write_text(report_text.replace('</body>', '\n'.join(extra_sections) + '\n</body>', 1), encoding='utf-8')
print('Combined report:', report_path)
print('Supplemental outputs:')
for output_path in sorted(TABLES_DIR.glob('supplemental_*')):
    print('-', output_path.name)
for output_path in sorted(FIGURES_DIR.glob('supplemental_*')):
    print('-', output_path.name)


## 25. Export the software and combined report
The ZIP contains the selected weights, configuration, app.py, dependencies, example ISIC images and evaluation outputs. It can also run locally using pip install -r requirements.txt, then python app.py.
The ZIP also contains the confidence-interval tables, optional comparison tables, Results text and analysis provenance from Sections 18–24.

In [ ]:
# Create a runnable deployment package containing the selected weights and config.
import importlib.metadata
EXAMPLES_DIR = OUTPUT_DIR / 'examples'
EXAMPLES_DIR.mkdir(exist_ok=True)
for outcome in ('TP', 'TN', 'FP', 'FN'):
    matching = test_predictions.loc[test_predictions.outcome == outcome]
    if len(matching):
        row = matching.iloc[0]
        shutil.copy2(row.image_path, EXAMPLES_DIR / (row.image_id + '.jpg'))
(OUTPUT_DIR / 'attention_pipeline_core.py').write_text('"""Inference and Gradio UI for the frozen Attention U-Net melanoma candidate.\n\nNo training or threshold selection occurs in this module. JPEG ROI encoding is\npreserved because the training notebook classified saved quality-95 JPEG ROIs.\n"""\nfrom __future__ import annotations\n\nfrom io import BytesIO\nfrom pathlib import Path\nimport hashlib\nimport json\nimport threading\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nfrom scipy import ndimage\nimport tensorflow as tf\n\nSEG_SIZE = (256, 256)\nCLS_SIZE = (224, 224)\n\n\ndef image_tensor(path, size, normalise=False):\n    pixels = tf.io.decode_image(tf.io.read_file(str(path)), channels=3,\n                                expand_animations=False)\n    pixels = tf.cast(tf.image.resize(pixels, size, method="bilinear"), tf.float32)\n    return pixels / 255.0 if normalise else pixels\n\n\ndef roi_tensor(roi):\n    # Match roi.save(..., quality=95) -> decode_classifier_image() in training.\n    buffer = BytesIO()\n    roi.convert("RGB").save(buffer, format="JPEG", quality=95)\n    pixels = tf.io.decode_jpeg(buffer.getvalue(), channels=3)\n    return tf.cast(tf.image.resize(pixels, CLS_SIZE), tf.float32)\n\n\ndef crop_from_prediction(image_pil, predicted_mask, padding=0.15):\n    image = image_pil.convert("RGB")\n    width, height = image.size\n    binary = Image.fromarray(np.uint8(np.asarray(predicted_mask) > 0) * 255)\n    binary = np.asarray(binary.resize(image.size, Image.Resampling.NEAREST)) > 0\n    labelled, count = ndimage.label(binary)\n    reason = "cropped"\n    if count == 0:\n        crop, reason = image, "empty predicted mask"\n    else:\n        sizes = np.bincount(labelled.ravel())\n        sizes[0] = 0\n        lesion = labelled == int(sizes.argmax())\n        area = int(lesion.sum())\n        if area < max(64, int(width * height * 0.001)) or area > width * height * 0.80:\n            crop, reason = image, "implausible predicted area"\n        else:\n            ys, xs = np.where(lesion)\n            span = max(np.ptp(xs) + 1, np.ptp(ys) + 1)\n            margin = int(np.ceil(span * padding))\n            x0 = max(0, int(xs.min()) - margin)\n            x1 = min(width, int(xs.max()) + margin + 1)\n            y0 = max(0, int(ys.min()) - margin)\n            y1 = min(height, int(ys.max()) + margin + 1)\n            crop = image.crop((x0, y0, x1, y1))\n    a = np.asarray(crop)\n    h, w = a.shape[:2]\n    side = max(h, w)\n    top, left = (side - h) // 2, (side - w) // 2\n    square = np.pad(a, ((top, side - h - top), (left, side - w - left), (0, 0)),\n                    mode="edge")\n    roi = Image.fromarray(square).resize((CLS_SIZE[1], CLS_SIZE[0]),\n                                         Image.Resampling.BILINEAR)\n    return roi, reason\n\n\ndef mask_overlay(image, mask):\n    base = np.asarray(image.convert("RGB"), dtype=np.float32)\n    mask = Image.fromarray(np.uint8(mask) * 255).resize(\n        image.size, Image.Resampling.NEAREST)\n    binary = np.asarray(mask) > 127\n    tint = np.array([35, 195, 145], dtype=np.float32)\n    base[binary] = 0.65 * base[binary] + 0.35 * tint\n    boundary = binary & ~ndimage.binary_erosion(binary)\n    base[boundary] = [255, 215, 30]\n    return Image.fromarray(np.uint8(np.clip(base, 0, 255)))\n\n\ndef gradcam_overlay(roi, heatmap):\n    from matplotlib import colormaps\n    image = roi.convert("RGB").resize((CLS_SIZE[1], CLS_SIZE[0]))\n    heat = Image.fromarray(np.uint8(np.clip(heatmap, 0, 1) * 255)).resize(\n        image.size, Image.Resampling.BILINEAR)\n    colors = colormaps["jet"](np.asarray(heat, dtype=np.float32) / 255.0)[..., :3]\n    base = np.asarray(image, dtype=np.float32) / 255.0\n    return Image.fromarray(np.uint8(255 * np.clip(0.62 * base + 0.38 * colors, 0, 1)))\n\n\nclass MelanomaPipeline:\n    def __init__(self, segmenter, classifier, configuration, run_dir=None):\n        self.segmenter = segmenter\n        self.classifier = classifier\n        self.configuration = dict(configuration)\n        self.run_dir = Path(run_dir) if run_dir else None\n        self.lock = threading.RLock()\n        cfg = self.configuration\n        if cfg.get("view") != "dual":\n            raise ValueError("This app requires the full-image + ROI classifier (view=\'dual\').")\n        for field in ("threshold", "segmentation_threshold"):\n            if field not in cfg or not 0 <= float(cfg[field]) <= 1:\n                raise ValueError(f"Missing or invalid saved configuration field: {field}")\n        for field in ("method", "segmentation_method"):\n            if cfg.get(field) not in ("single", "4-flip average"):\n                raise ValueError(f"Unsupported inference method: {cfg.get(field)}")\n        if segmenter.input_shape[1:] != (*SEG_SIZE, 3):\n            raise ValueError("Segmenter input must be 256 x 256 RGB.")\n        if len(classifier.inputs) != 2:\n            raise ValueError("Expected two classifier inputs: full image and ROI.")\n        shapes = [tuple(x.shape[1:]) for x in classifier.inputs]\n        if shapes != [(*CLS_SIZE, 3), (*CLS_SIZE, 3)]:\n            raise ValueError(f"Classifier input shapes are incompatible: {shapes}")\n\n    @classmethod\n    def load(cls, run_dir):\n        run_dir = Path(run_dir)\n        paths = {\n            "configuration": run_dir / "tables" / "selected_configuration.json",\n            "segmenter": run_dir / "models" / "attention_unet_best.keras",\n            "classifier": run_dir / "models" / "selected_final_classifier.keras",\n        }\n        missing = [str(p) for p in paths.values() if not p.is_file()]\n        if missing:\n            raise FileNotFoundError("The saved pipeline is incomplete. Missing:\\n" + "\\n".join(missing))\n        configuration = json.loads(paths["configuration"].read_text())\n        provenance = run_dir / "tables" / "segmentation_configuration.json"\n        if provenance.is_file():\n            expected = json.loads(provenance.read_text()).get("checkpoint_sha256")\n            if expected:\n                actual = hashlib.sha256(paths["segmenter"].read_bytes()).hexdigest()\n                if actual != expected:\n                    raise ValueError("The segmentation checkpoint does not match its saved configuration.")\n        manifest_path = run_dir / "deployment_manifest.json"\n        if manifest_path.is_file():\n            for relative, expected in json.loads(manifest_path.read_text()).get("sha256", {}).items():\n                actual = hashlib.sha256((run_dir / relative).read_bytes()).hexdigest()\n                if actual != expected:\n                    raise ValueError(f"Deployment bundle file has changed: {relative}")\n        # compile=False avoids needing custom training losses for inference.\n        segmenter = tf.keras.models.load_model(paths["segmenter"], compile=False)\n        classifier = tf.keras.models.load_model(paths["classifier"], compile=False)\n        return cls(segmenter, classifier, configuration, run_dir)\n\n    def segmentation_probabilities(self, images, batch_size=4):\n        images = np.asarray(images, dtype=np.float32)\n        methods = [(), (2,), (1,), (1, 2)] if self.configuration[\n            "segmentation_method"] == "4-flip average" else [()]\n        predictions = []\n        for axes in methods:\n            transformed = np.flip(images, axis=axes) if axes else images\n            score = self.segmenter.predict(np.ascontiguousarray(transformed),\n                                            batch_size=batch_size, verbose=0)[..., 0]\n            score = np.flip(score, axis=axes) if axes else score\n            predictions.append(score)\n        return np.mean(np.stack(predictions), axis=0)\n\n    def classifier_probabilities(self, full_images, roi_images, batch_size=8):\n        full_images = np.asarray(full_images, dtype=np.float32)\n        roi_images = np.asarray(roi_images, dtype=np.float32)\n        methods = [(), (2,), (1,), (1, 2)] if self.configuration[\n            "method"] == "4-flip average" else [()]\n        predictions = []\n        for axes in methods:\n            full = np.flip(full_images, axis=axes) if axes else full_images\n            roi = np.flip(roi_images, axis=axes) if axes else roi_images\n            features = {"full_image": np.ascontiguousarray(full),\n                        "roi_image": np.ascontiguousarray(roi)}\n            predictions.append(self.classifier.predict(\n                features, batch_size=batch_size, verbose=0).ravel())\n        return np.mean(np.stack(predictions), axis=0)\n\n    def gradcam(self, full_batch, roi_batch, target_class):\n        model = self.classifier\n        backbone = model.get_layer("efficientnetb0")\n        gap = model.get_layer("gap")\n        with tf.GradientTape() as tape:\n            full_map = backbone(full_batch, training=False)\n            roi_map = backbone(roi_batch, training=False)\n            tape.watch(roi_map)\n            features = model.get_layer("combine_features")([gap(full_map), gap(roi_map)])\n            hidden = model.get_layer("hidden_dense")(features)\n            hidden = model.get_layer("head_dropout")(hidden, training=False)\n            output = model.get_layer("prediction")\n            logit = tf.linalg.matmul(hidden, output.kernel) + output.bias\n            score = logit[:, 0] if target_class == 1 else -logit[:, 0]\n        gradients = tape.gradient(score, roi_map)\n        if gradients is None:\n            raise RuntimeError("Grad-CAM could not trace the ROI feature map.")\n        weights = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)\n        heatmap = tf.nn.relu(tf.reduce_sum(roi_map * weights, axis=-1)[0])\n        return (heatmap / (tf.reduce_max(heatmap) + 1e-8)).numpy()\n\n    def predict_file(self, path, explain=True):\n        if path is None:\n            raise ValueError("Upload a dermoscopic lesion image first.")\n        path = Path(path)\n        with Image.open(path) as handle:\n            original = handle.convert("RGB")\n        with self.lock:\n            segmentation_input = image_tensor(path, SEG_SIZE, normalise=True)[None, ...]\n            probabilities = self.segmentation_probabilities(segmentation_input)[0]\n            mask = probabilities >= float(self.configuration["segmentation_threshold"])\n            roi, reason = crop_from_prediction(\n                original, mask, padding=float(self.configuration.get("roi_padding", 0.15)))\n            full_input = image_tensor(path, CLS_SIZE)[None, ...]\n            roi_input = roi_tensor(roi)[None, ...]\n            score = float(self.classifier_probabilities(full_input, roi_input)[0])\n            predicted = int(score >= float(self.configuration["threshold"]))\n            heatmap = self.gradcam(full_input, roi_input, predicted) if explain else None\n        result = {\n            "original": original, "mask": Image.fromarray(np.uint8(mask) * 255),\n            "mask_overlay": mask_overlay(original, mask), "roi": roi,\n            "score": score, "predicted_label": predicted,\n            "label": "Melanoma" if predicted else "Benign",\n            "roi_reason": reason, "probability_map": probabilities,\n        }\n        result["gradcam"] = gradcam_overlay(roi, heatmap) if heatmap is not None else None\n        return result\n\n\ndef build_demo(pipeline, sample_paths=()):\n    import gradio as gr\n    cfg = pipeline.configuration\n\n    def preview(path):\n        if path is None:\n            return None\n        with Image.open(path) as image:\n            return image.convert("RGB")\n\n    def analyse(path):\n        try:\n            result = pipeline.predict_file(path)\n        except (ValueError, FileNotFoundError, OSError) as error:\n            raise gr.Error(str(error)) from error\n        summary = f"### Model prediction: **{result[\'label\']}**\\n"\n        summary += (f"Melanoma model score: **{result[\'score\']:.4f}**. "\n                    f"Decision threshold: {float(cfg[\'threshold\']):.4f}.\\n\\n"\n                    "The score is a model output, not a calibrated clinical probability.")\n        if result["roi_reason"] != "cropped":\n            summary += "\\n\\nThe lesion crop was unreliable; the classifier used the full image for the ROI view."\n        panels = [(result["original"], "Original image"),\n                  (result["mask"], "Predicted lesion mask"),\n                  (result["mask_overlay"], "Lesion outline"),\n                  (result["roi"], "Lesion ROI"),\n                  (result["gradcam"], "Grad-CAM on ROI")]\n        return summary, panels\n\n    def clear():\n        return None, None, "Upload a dermoscopic lesion image, then select Analyse.", []\n\n    with gr.Blocks(title="Attention U-Net melanoma research demo",\n                   delete_cache=(3600, 3600)) as demo:\n        gr.Markdown("# Melanoma image analysis\\n"\n                    "Attention U-Net lesion segmentation with EfficientNetB0 classification.\\n\\n"\n                    "**Student research prototype. Not validated for clinical diagnosis.**")\n        with gr.Tab("Analyse image"):\n            with gr.Row():\n                with gr.Column(scale=1):\n                    upload = gr.File(label="Upload dermoscopic image (JPG or PNG)",\n                                     file_types=[".jpg", ".jpeg", ".png"], type="filepath")\n                    input_preview = gr.Image(label="Uploaded image", interactive=False)\n                    with gr.Row():\n                        analyse_button = gr.Button("Analyse", variant="primary")\n                        clear_button = gr.Button("Clear")\n                    if sample_paths:\n                        gr.Examples(examples=[[str(p)] for p in sample_paths],\n                                    inputs=upload, label="Public ISIC example images")\n                with gr.Column(scale=2):\n                    summary = gr.Markdown("Upload a dermoscopic lesion image, then select Analyse.")\n                    panels = gr.Gallery(label="Pipeline outputs", columns=3,\n                                        object_fit="contain", height="auto")\n                    gr.Markdown("The lesion mask estimates the lesion boundary. "\n                                "Grad-CAM shows image regions contributing to the model prediction; "\n                                "it is not a lesion mask.")\n            upload.change(preview, inputs=upload, outputs=input_preview)\n            analyse_button.click(analyse, inputs=upload, outputs=[summary, panels],\n                                 api_name="analyse", concurrency_limit=1)\n            clear_button.click(clear, outputs=[upload, input_preview, summary, panels])\n        with gr.Tab("Evaluation"):\n            gr.Markdown("## Saved evaluation results\\n"\n                        "Classification and segmentation measure different tasks. "\n                        "The official ISIC test set was viewed in earlier experiments; "\n                        "these results are exploratory follow-up.")\n            run_dir = pipeline.run_dir\n            displayed = False\n            if run_dir:\n                for filename, label in [("complete_segmentation_test_metrics.csv", "Lesion segmentation"),\n                                        ("complete_classification_test_metrics.csv", "Melanoma classification")]:\n                    path = run_dir / "tables" / filename\n                    if path.is_file():\n                        gr.Markdown("### " + label)\n                        gr.Dataframe(value=pd.read_csv(path).T.reset_index(),\n                                     headers=["Metric", "Value"], interactive=False)\n                        displayed = True\n                figures = sorted((run_dir / "figures").glob("*.png"))\n                if figures:\n                    gr.Gallery(value=[(str(p), p.stem.replace("_", " ")) for p in figures],\n                               label="Evaluation figures", columns=1, object_fit="contain")\n                    displayed = True\n            if not displayed:\n                gr.Markdown("Run the reporting notebook to add test loss, metrics and evaluation figures.")\n    return demo\n', encoding='utf-8')
(OUTPUT_DIR / 'app.py').write_text('from pathlib import Path\nimport os\nfrom attention_pipeline_core import MelanomaPipeline, build_demo\n\nrun_dir = Path(os.environ.get("MELANOMA_RUN_DIR", Path(__file__).resolve().parent))\npipeline = MelanomaPipeline.load(run_dir)\nexamples = sorted((run_dir / "examples").glob("*.jpg"))\ndemo = build_demo(pipeline, examples)\nif __name__ == "__main__":\n    demo.queue(default_concurrency_limit=1).launch(\n        server_name="0.0.0.0", server_port=int(os.environ.get("PORT", "7860")),\n        share=False, show_error=True,\n        allowed_paths=[str(run_dir / "figures"), str(run_dir / "examples")],\n    )\n', encoding='utf-8')
(OUTPUT_DIR / 'requirements.txt').write_text('tensorflow==2.21.0\ngradio==6.30.0\nnumpy>=1.26,<3\npandas>=2,<4\nscipy>=1.11,<2\nmatplotlib>=3.8,<4\npillow>=10,<13\nscikit-learn>=1.3,<2\n', encoding='utf-8')
protected = ['models/attention_unet_best.keras', 'models/selected_final_classifier.keras',
             'tables/selected_configuration.json']
(OUTPUT_DIR / 'deployment_manifest.json').write_text(json.dumps({
    'sha256': {p: hashlib.sha256((OUTPUT_DIR / p).read_bytes()).hexdigest() for p in protected},
    'evaluation': 'Exploratory follow-up on a previously viewed official test set.',
    'versions': {'tensorflow': tf.__version__,
                 **{name: importlib.metadata.version(name) for name in
                    ['keras', 'gradio', 'numpy', 'pillow']}},
}, indent=2))
bundle_path = OUTPUT_DIR / 'melanoma_software_bundle.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for item in [OUTPUT_DIR / 'app.py', OUTPUT_DIR / 'attention_pipeline_core.py',
                 OUTPUT_DIR / 'requirements.txt', OUTPUT_DIR / 'deployment_manifest.json', report_path]:
        archive.write(item, item.relative_to(OUTPUT_DIR))
    for folder in ['tables', 'figures', 'examples']:
        for item in sorted((OUTPUT_DIR / folder).rglob('*')):
            if item.is_file():
                archive.write(item, item.relative_to(OUTPUT_DIR))
    for name in ['attention_unet_best.keras', 'selected_final_classifier.keras']:
        item = MODELS_DIR / name
        archive.write(item, item.relative_to(OUTPUT_DIR))
print('All results saved:', OUTPUT_DIR)
print('Software bundle:', bundle_path)
from IPython.display import FileLink, display
display(FileLink(str(report_path)))
display(FileLink(str(bundle_path)))


## 26. Launch the Gradio application
Open the printed gradio.live link on your computer or phone. The link is temporary and requires an active Colab runtime. Analyse an image to see the prediction, mask, ROI and Grad-CAM.

In [ ]:
# Launch the real models as an interactive software demo.
if 'demo' in globals():
    demo.close()
demo = build_demo(pipeline, sorted(EXAMPLES_DIR.glob('*.jpg')))
demo.queue(default_concurrency_limit=1).launch(
    share=SHARE_APP, debug=False, show_error=True,
    allowed_paths=[str(FIGURES_DIR), str(EXAMPLES_DIR)],
)
